In [ ]:
# Block 1: notebook description and analysis objective

#This notebook is being used to evaluate the distribution shape, tail risk, and volatility behavior of a single asset.
#Original Risk Analysis blocks included here: 6-10.


In [ ]:
import logging
import warnings
from pathlib import Path
import runpy

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import yfinance as yf
from plotly.subplots import make_subplots

from Quantapp.visualization import (
    BarChartPlotter,
    CandleStickPlotter,
    LineChartPlotter,
    Plotter,
    build_time_range_buttons,)
from Quantapp.analytics import (
    Helper,
    Models,
    MomentumAnalytics,
    RiskDistributionAnalytics,
    RiskRelativeAnalytics,
    SeriesTransforms,
    TimeSeriesAnalytics as Rolling,)
from Quantapp.data import (
    MacroDataClient,
    align_series_to_common_index,
    load_benchmark_data,
    normalize_benchmark_tickers,)

warnings.filterwarnings("ignore")
logger = logging.getLogger("yfinance")
rolling = Rolling()
series_transforms = SeriesTransforms()
momentum_analytics = MomentumAnalytics()
risk_distribution_analytics = RiskDistributionAnalytics()
risk_relative_analytics = RiskRelativeAnalytics()

In [ ]:
qp = Plotter()
qe = MacroDataClient()
helper = Helper()
model = Models()
lineChartPlotter = LineChartPlotter()
candleStickPlotter = CandleStickPlotter()
barChartPlotter = BarChartPlotter()

PARAMS_FILE_CANDIDATES = []
for base_path in (Path.cwd(), *Path.cwd().parents):
    PARAMS_FILE_CANDIDATES.extend([
        base_path / "params.py",
        base_path / "Notebooks" / "Single Asset Profile" / "params.py",
    ])

seen_params_paths = set()
for params_file in PARAMS_FILE_CANDIDATES:
    params_file = params_file.resolve()
    if params_file in seen_params_paths:
        continue
    seen_params_paths.add(params_file)
    if params_file.exists():
        break
else:
    raise FileNotFoundError("Could not locate params.py in the Single Asset Profile workspace.")

notebook_params = runpy.run_path(str(params_file))
get_single_asset_profile_params = notebook_params["get_single_asset_profile_params"]
resolve_time_frame_map = notebook_params["resolve_time_frame_map"]


PLOTLY_NOTEBOOK_CONFIG = {"responsive": True, "scrollZoom": True}
for renderer_name in ("plotly_mimetype", "notebook", "notebook_connected", "jupyterlab"):
    try:
        pio.renderers[renderer_name].config = PLOTLY_NOTEBOOK_CONFIG.copy()
    except Exception:
        pass

def show_plotly_figure(fig, *, config=None, **layout_kwargs):
    merged_config = PLOTLY_NOTEBOOK_CONFIG.copy()
    if config:
        merged_config.update(config)
    fig.update_layout(autosize=True, **layout_kwargs)
    fig.show(config=merged_config)

In [ ]:
# Block 3: load shared analysis parameters
# Edit Notebooks/Single Asset Profile/params.py to update these defaults for every notebook.
single_asset_params = get_single_asset_profile_params()

ticker_str = single_asset_params["ticker_str"]
interval = single_asset_params["interval"]
period = single_asset_params["period"]
risk_free_ticker = single_asset_params["risk_free_ticker"]
benchmark_tickers = list(single_asset_params["benchmark_tickers"])
trading_strategy = single_asset_params["trading_strategy"]
length_of_plots = single_asset_params["length_of_plots"]
var_position_value = single_asset_params["var_position_value"]

single_asset_params

In [ ]:
# Block 4: organize structural, position, swing timeframe lists

strategy = str(trading_strategy).strip().lower()
time_frame_map = resolve_time_frame_map(strategy)
time_frame_short = time_frame_map["short"]
time_frame_mid = time_frame_map["mid"]
time_frame_long = time_frame_map["long"]

return_frequencies = ('monthly', 'weekly', 'daily')


In [ ]:
# Block 5: load and align asset, benchmark, and return data

# Download and normalize asset-level data
ticker = yf.Ticker(ticker_str).history(period=period, interval=interval)
vix = yf.Ticker('^VIX').history(period=period, interval=interval)
risk_free_proxy = yf.Ticker(risk_free_ticker).history(period=period, interval=interval)
ticker = helper.simplify_datetime_index(ticker)
ticker_trade_range_source = helper.build_equity_like_trade_range_source(ticker_str, ticker)
vix = helper.simplify_datetime_index(vix)
risk_free_proxy = helper.simplify_datetime_index(risk_free_proxy)
if risk_free_proxy.empty or 'Close' not in risk_free_proxy:
    raise ValueError(f"No risk-free history available for {risk_free_ticker}.")
risk_free_annual_yield = risk_free_proxy['Close'].dropna().sort_index().div(100)
risk_free_daily_rate = ((1 + risk_free_annual_yield) ** (1 / 252) - 1).shift(1)

# Download benchmark data once and keep it in collections for downstream cells
benchmark_tickers = normalize_benchmark_tickers(benchmark_tickers, ticker_str, include_asset=True)
benchmark_data, skipped_benchmarks = load_benchmark_data(benchmark_tickers, period, interval, helper)
if skipped_benchmarks:
    print(f'Skipped benchmarks with no data: {skipped_benchmarks}')

analysis_index, ticker, vix, benchmark_data = align_series_to_common_index(ticker, vix, benchmark_data)
risk_free_daily_rate = risk_free_daily_rate.reindex(ticker.index).ffill()

# Calculate asset and benchmark returns for the frequencies used elsewhere in the notebook
ticker_returns = {frequency: series_transforms.returns(ticker, frequency=frequency) for frequency in return_frequencies}
ticker_monthly_returns = ticker_returns['monthly']
ticker_weekly_returns = ticker_returns['weekly']
ticker_daily_returns = ticker_returns['daily']

benchmark_returns = {
    symbol: {frequency: series_transforms.returns(frame, frequency=frequency) for frequency in return_frequencies}
    for symbol, frame in benchmark_data.items()
}

vix_returns = {frequency: series_transforms.returns(vix, frequency=frequency) for frequency in return_frequencies}
vix_monthly_returns = vix_returns['monthly']
vix_weekly_returns = vix_returns['weekly']
vix_daily_returns = vix_returns['daily']


In [ ]:
# Block 6: analyze daily returns, skew, kurtosis, and Gini z-scores

distribution_window_options = [21, 50, 200]

distribution_context = risk_distribution_analytics.build_risk_distribution_context(
    close_series=ticker['Close'],
    windows=distribution_window_options,
    default_window=200 if 200 in distribution_window_options else max(distribution_window_options),
)

fig = lineChartPlotter.plot_distribution_shape_zscores(
    metrics_by_window=distribution_context['metrics_by_window'],
    window_options=distribution_context['windows'],
    default_window=distribution_context['default_window'],
    ticker_label=ticker_str,
    include_return_panel=False,
)
show_plotly_figure(fig)


In [ ]:
# Block 7: compare close-to-close trade-range methods with Dash selectors


# Refresh the shared plotter so notebook reruns pick up local code edits without a kernel restart.
from importlib import reload
import Quantapp.visualization.line_chart_plotter as _line_chart_plotter_module
reload(_line_chart_plotter_module)
lineChartPlotter = _line_chart_plotter_module.LineChartPlotter()

trade_range_price_frame = globals().get('ticker_trade_range_source', ticker)[['Open', 'Close']].dropna().copy()
trade_range_window_candidates = [21, 50, 200]
trade_range_max_supported_window = max(1, len(trade_range_price_frame) - 1)
trade_range_window_options = [window for window in trade_range_window_candidates if window <= trade_range_max_supported_window]
if not trade_range_window_options:
    raise ValueError(
        f'Trade-range analysis needs at least 21 completed sessions. '
        f'Only {trade_range_max_supported_window} are available for {ticker_str} '
        f'using {trade_range_price_frame.attrs.get("session_mode", "the current session mode")}.'
    )
trade_range_default_window = 50 if 50 in trade_range_window_options else max(trade_range_window_options)
trade_range_default_horizon = 1
trade_range_interval_levels = [0.85, 0.95, 0.99]
trade_range_tail_levels = [0.85, 0.95, 0.99]
trade_range_confidence_options = sorted(set(trade_range_interval_levels).union(trade_range_tail_levels))
trade_range_default_confidence = 0.95 if 0.95 in trade_range_confidence_options else trade_range_confidence_options[0]

trade_range_history_context = risk_distribution_analytics.build_trade_range_history_context(
    price_frame=trade_range_price_frame,
    windows=trade_range_window_options,
    horizon_sessions=trade_range_default_horizon,
    interval_confidence_levels=trade_range_interval_levels,
    tail_confidence_levels=trade_range_tail_levels,
    default_window=trade_range_default_window,
)
trade_range_history_contexts_by_horizon = {
    int(trade_range_default_horizon): trade_range_history_context,
}

trade_range_history_fig = lineChartPlotter.plot_trade_range_history_profile(
    history_context=trade_range_history_context,
    ticker_label=ticker_str,
)
from plotly.subplots import make_subplots
import copy as _copy


def _axis_index_from_ref(axis_ref, axis_letter):
    if axis_ref in (None, axis_letter):
        return 1
    if isinstance(axis_ref, str) and axis_ref.startswith(axis_letter):
        digits = ''.join(ch for ch in axis_ref[1:] if ch.isdigit())
        return int(digits) if digits else 1
    return 1


def _shift_axis_ref(axis_ref, row_offset):
    if not isinstance(axis_ref, str) or axis_ref == 'paper':
        return axis_ref
    suffix = ' domain' if axis_ref.endswith(' domain') else ''
    base = axis_ref[:-7] if suffix else axis_ref
    if not base or base[0] not in ('x', 'y'):
        return axis_ref
    axis_letter = base[0]
    digits = base[1:]
    axis_index = int(digits) if digits else 1
    shifted_index = axis_index + row_offset
    shifted_base = axis_letter if shifted_index == 1 else f'{axis_letter}{shifted_index}'
    return shifted_base + suffix


def _compose_trade_range_stack(history_fig, cone_fig, *, title_text):
    def _figure_row_count(fig):
        row_count = max(
            (_axis_index_from_ref(getattr(trace, 'yaxis', None), 'y') for trace in fig.data),
            default=0,
        )
        if row_count <= 0:
            raise ValueError('Trade-range figure does not contain any subplot traces to stack.')
        return row_count

    def _subplot_titles_from_figure(fig, count):
        titles = [
            str(getattr(annotation, 'text', ''))
            for annotation in list(getattr(fig.layout, 'annotations', []))[:count]
        ]
        if len(titles) != count:
            raise ValueError('Trade-range figure is missing expected subplot titles.')
        return tuple(titles)

    def _to_json(item):
        payload = item.to_plotly_json() if hasattr(item, 'to_plotly_json') else dict(item)
        return _copy.deepcopy(payload)

    def _shift_shapes(shapes, row_offset):
        shifted_shapes = []
        for shape in shapes:
            shape_payload = _to_json(shape)
            shape_payload['xref'] = _shift_axis_ref(shape_payload.get('xref'), row_offset)
            shape_payload['yref'] = _shift_axis_ref(shape_payload.get('yref'), row_offset)
            shifted_shapes.append(shape_payload)
        return shifted_shapes

    def _shift_annotations(annotations, row_offset):
        shifted_annotations = []
        for annotation in annotations:
            annotation_payload = _to_json(annotation)
            annotation_payload['xref'] = _shift_axis_ref(annotation_payload.get('xref'), row_offset)
            annotation_payload['yref'] = _shift_axis_ref(annotation_payload.get('yref'), row_offset)
            shifted_annotations.append(annotation_payload)
        return shifted_annotations

    def _window_title(base_title, window_label):
        suffix = '-Session Lookback)'
        marker = base_title.rfind('(')
        suffix_index = base_title.rfind(suffix)
        if marker != -1 and suffix_index != -1 and marker < suffix_index:
            return f"{base_title[:marker + 1]}{window_label}{suffix}"
        return base_title

    def _layout_axis(fig, axis_letter, axis_index):
        axis_name = f'{axis_letter}axis' if axis_index == 1 else f'{axis_letter}axis{axis_index}'
        return getattr(fig.layout, axis_name, None)

    def _coerce_layout_value(value):
        if value is None or isinstance(value, str):
            return value
        if isinstance(value, (list, tuple)):
            return list(value)
        if hasattr(value, 'tolist'):
            try:
                return value.tolist()
            except Exception:
                return value
        return value

    def _axis_ref(axis_letter, axis_index):
        return axis_letter if axis_index == 1 else f'{axis_letter}{axis_index}'

    def _datetime_bounds(fig):
        bounds = []
        for trace in fig.data:
            x_values = getattr(trace, 'x', None)
            if x_values is None:
                continue
            try:
                x_index = pd.to_datetime(pd.Index(x_values), errors='coerce')
            except Exception:
                continue
            if len(x_index) == 0:
                continue
            x_index = x_index[~pd.isna(x_index)]
            if len(x_index) == 0:
                continue
            bounds.append((x_index.min(), x_index.max()))
        if not bounds:
            return None, None
        return min(start for start, _ in bounds), max(end for _, end in bounds)

    def _history_timeframe_buttons(global_start, global_end):
        def _timeframe_range(years=None):
            start = global_start if years is None else max(global_start, global_end - pd.DateOffset(years=years))
            layout_updates = {}
            for axis_index in range(1, history_row_count + 1):
                axis_name = 'xaxis' if axis_index == 1 else f'xaxis{axis_index}'
                layout_updates[f'{axis_name}.range'] = [start, global_end]
            return layout_updates

        return [
            dict(label='1Y', method='relayout', args=[_timeframe_range(1)]),
            dict(label='2Y', method='relayout', args=[_timeframe_range(2)]),
            dict(label='3Y', method='relayout', args=[_timeframe_range(3)]),
            dict(label='5Y', method='relayout', args=[_timeframe_range(5)]),
            dict(label='10Y', method='relayout', args=[_timeframe_range(10)]),
            dict(label='Full', method='relayout', args=[_timeframe_range(None)]),
        ]

    history_row_count = _figure_row_count(history_fig)
    cone_row_count = _figure_row_count(cone_fig)
    total_rows = history_row_count + cone_row_count

    history_row_heights = [0.14, 0.12, 0.11, 0.11, 0.11][:history_row_count]
    if len(history_row_heights) < history_row_count:
        history_row_heights.extend([0.11] * (history_row_count - len(history_row_heights)))
    cone_height_budget = max(0.20, 1.0 - sum(history_row_heights))
    if cone_row_count <= 1:
        cone_row_heights = [cone_height_budget]
    else:
        cone_panel_count = cone_row_count - 1
        cone_distribution_height = cone_height_budget * 0.32
        cone_panel_height = (cone_height_budget - cone_distribution_height) / cone_panel_count
        cone_row_heights = [cone_panel_height] * cone_panel_count + [cone_distribution_height]

    combined_fig = make_subplots(
        rows=total_rows,
        cols=1,
        shared_xaxes=False,
        vertical_spacing=0.04,
        row_heights=history_row_heights + cone_row_heights,
        subplot_titles=(
            _subplot_titles_from_figure(history_fig, history_row_count)
            + _subplot_titles_from_figure(cone_fig, cone_row_count)
        ),
    )
    subplot_title_annotations = [_copy.deepcopy(annotation).to_plotly_json() for annotation in combined_fig.layout.annotations]

    def _copy_axis_layout(source_fig, source_row, target_row):
        source_xaxis = _layout_axis(source_fig, 'x', source_row)
        if source_xaxis is not None:
            x_kwargs = {}
            for attr in ('range', 'tickmode', 'tickvals', 'ticktext', 'ticksuffix', 'tickprefix'):
                value = getattr(source_xaxis, attr, None)
                if value is not None:
                    x_kwargs[attr] = _coerce_layout_value(value)
            title_value = getattr(getattr(source_xaxis, 'title', None), 'text', None)
            if title_value is not None:
                x_kwargs['title_text'] = title_value
            if x_kwargs:
                combined_fig.update_xaxes(row=target_row, col=1, **x_kwargs)

        source_yaxis = _layout_axis(source_fig, 'y', source_row)
        if source_yaxis is not None:
            y_kwargs = {}
            for attr in ('range', 'tickprefix', 'tickformat', 'ticksuffix'):
                value = getattr(source_yaxis, attr, None)
                if value is not None:
                    y_kwargs[attr] = _coerce_layout_value(value)
            title_value = getattr(getattr(source_yaxis, 'title', None), 'text', None)
            if title_value is not None:
                y_kwargs['title_text'] = title_value
            if y_kwargs:
                combined_fig.update_yaxes(row=target_row, col=1, **y_kwargs)

    for trace in history_fig.data:
        target_row = _axis_index_from_ref(getattr(trace, 'yaxis', None), 'y')
        stacked_trace = _copy.deepcopy(trace)
        stacked_trace.xaxis = None
        stacked_trace.yaxis = None
        combined_fig.add_trace(stacked_trace, row=target_row, col=1)

    for trace in cone_fig.data:
        target_row = history_row_count + _axis_index_from_ref(getattr(trace, 'yaxis', None), 'y')
        stacked_trace = _copy.deepcopy(trace)
        stacked_trace.xaxis = None
        stacked_trace.yaxis = None
        combined_fig.add_trace(stacked_trace, row=target_row, col=1)

    history_static_shapes = _shift_shapes(history_fig.layout.shapes, 0)
    history_extra_annotations = _shift_annotations(history_fig.layout.annotations[history_row_count:], 0)
    cone_default_shapes = _shift_shapes(cone_fig.layout.shapes, history_row_count)
    cone_default_annotations = _shift_annotations(cone_fig.layout.annotations[cone_row_count:], history_row_count)

    for source_row in range(1, history_row_count + 1):
        _copy_axis_layout(history_fig, source_row, source_row)
    for source_row in range(1, cone_row_count + 1):
        _copy_axis_layout(cone_fig, source_row, history_row_count + source_row)

    if history_row_count > 1:
        history_anchor_ref = _axis_ref('x', history_row_count)
        for row in range(1, history_row_count):
            combined_fig.update_xaxes(matches=history_anchor_ref, row=row, col=1)

    cone_path_row_count = max(0, cone_row_count - 1)
    if cone_path_row_count > 1:
        cone_anchor_row = history_row_count + cone_path_row_count
        cone_anchor_ref = _axis_ref('x', cone_anchor_row)
        for row in range(history_row_count + 1, cone_anchor_row):
            combined_fig.update_xaxes(matches=cone_anchor_ref, row=row, col=1)

    updatemenus = []
    static_header_annotations = []
    history_menu = history_fig.layout.updatemenus[0] if getattr(history_fig.layout, 'updatemenus', None) else None
    cone_menu = cone_fig.layout.updatemenus[0] if getattr(cone_fig.layout, 'updatemenus', None) else None
    active_index = 0
    initial_title_text = title_text

    if history_menu is not None and cone_menu is not None:
        history_buttons = list(history_menu.buttons)
        cone_buttons = list(cone_menu.buttons)
        if len(history_buttons) == len(cone_buttons) and history_buttons:
            active_index = int(history_menu.active) if history_menu.active is not None else 0
            active_index = max(0, min(active_index, len(history_buttons) - 1))
            combined_buttons = []

            for history_button, cone_button in zip(history_buttons, cone_buttons):
                history_label = str(history_button.label)
                cone_label = str(cone_button.label)
                if history_label != cone_label:
                    raise ValueError('History and cone dropdown labels do not align for stacked trade-range figure.')

                history_args = history_button.args if history_button.args is not None else []
                cone_args = cone_button.args if cone_button.args is not None else []
                history_visibility = list(history_args[0].get('visible', [])) if history_args else []
                cone_visibility = list(cone_args[0].get('visible', [])) if cone_args else []
                if not history_visibility:
                    history_visibility = [trace.visible if trace.visible is not None else True for trace in history_fig.data]
                if not cone_visibility:
                    cone_visibility = [trace.visible if trace.visible is not None else True for trace in cone_fig.data]

                cone_layout_updates = cone_args[1] if len(cone_args) > 1 else {}
                cone_shapes = _shift_shapes(
                    cone_layout_updates.get('shapes', cone_fig.layout.shapes),
                    history_row_count,
                )
                cone_annotations = _shift_annotations(
                    cone_layout_updates.get('annotations', cone_fig.layout.annotations)[cone_row_count:],
                    history_row_count,
                )
                combined_buttons.append(
                    dict(
                        label=history_label,
                        method='update',
                        args=[
                            {'visible': history_visibility + cone_visibility},
                            {
                                'title': lineChartPlotter._header_title(_window_title(title_text, history_label)),
                                'shapes': history_static_shapes + cone_shapes,
                                'annotations': subplot_title_annotations + history_extra_annotations + static_header_annotations + cone_annotations,
                            },
                        ],
                    )
                )

            initial_title_text = _window_title(title_text, str(history_buttons[active_index].label))
            updatemenus.append(
                lineChartPlotter._dropdown_menu(
                    buttons=combined_buttons,
                    x=0.0,
                    active=active_index,
                )
            )

    history_global_start, history_global_end = _datetime_bounds(history_fig)
    if history_global_start is not None and history_global_end is not None:
        timeframe_menu_x = 0.18 if updatemenus else 0.0
        updatemenus.append(
            lineChartPlotter._dropdown_menu(
                buttons=_history_timeframe_buttons(history_global_start, history_global_end),
                x=timeframe_menu_x,
                active=2,
            )
        )
        static_header_annotations.append(
            dict(
                text='View timeframe',
                x=timeframe_menu_x,
                xref='paper',
                y=1.115,
                yref='paper',
                showarrow=False,
                xanchor='left',
            )
        )

    combined_fig.update_layout(
        title=lineChartPlotter._header_title(initial_title_text),
        height=2575 if cone_row_count <= 2 else 2875,
        margin=lineChartPlotter._header_margin(top=205),
        template='plotly_dark',
        hovermode='closest',
        legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='left', x=0),
        bargap=0.08,
        updatemenus=updatemenus,
        shapes=history_static_shapes + cone_default_shapes,
        annotations=subplot_title_annotations + history_extra_annotations + static_header_annotations + cone_default_annotations,
    )
    return combined_fig


def _stack_trade_range_figures(history_fig, cone_fig, *, title_text):
    return _compose_trade_range_stack(
        history_fig,
        cone_fig,
        title_text=title_text,
    )

trade_range_cone_contexts_by_key = {
    (int(trade_range_default_window), int(trade_range_default_horizon)): (
        risk_distribution_analytics.build_trade_range_probability_context(
            price_frame=trade_range_price_frame,
            window=int(trade_range_default_window),
            horizon_sessions=int(trade_range_default_horizon),
            interval_confidence_levels=trade_range_interval_levels,
            tail_confidence_levels=trade_range_tail_levels,
        )
    )
}

trade_range_window = trade_range_history_context['default_window']
trade_range_horizon = int(trade_range_default_horizon)
trade_range_context = trade_range_cone_contexts_by_key[(trade_range_window, trade_range_horizon)]

range_summary = trade_range_context['range_summary_table'].copy()
tail_summary = trade_range_context['tail_summary_table'].copy()

range_summary = range_summary.rename(columns={
    'Lower Close': 'Lower Exit Price',
    'Upper Close': 'Upper Exit Price',
})
tail_summary = tail_summary.rename(columns={
    'Confidence': 'Tail Confidence',
})

if not range_summary.empty:
    for column in ('Lower Return', 'Upper Return'):
        if column in range_summary.columns:
            range_summary[column] = range_summary[column].map(
                lambda value: f'{value:.2%}' if pd.notna(value) else value
            )
    for column in ('Lower Exit Price', 'Upper Exit Price'):
        if column in range_summary.columns:
            range_summary[column] = range_summary[column].map(
                lambda value: f'${value:,.2f}' if pd.notna(value) else value
            )

if not tail_summary.empty:
    for column in ('Long VaR Floor', 'Long CVaR Floor', 'Short VaR Ceiling', 'Short CVaR Ceiling'):
        if column in tail_summary.columns:
            tail_summary[column] = tail_summary[column].map(
                lambda value: f'${value:,.2f}' if pd.notna(value) else value
            )

# Build the optional GARCH(1,1)-normal variant so it can share the same selector.
for _garch_output_name in (
    'garch_trade_history_context',
    'formatted_garch_model_summary',
    'garch_range_summary',
    'garch_tail_summary',
    'garch_trade_combined_fig',
    'garch_trade_range_fig',
    'garch_cone_components_by_key',
    'garch_trade_history_contexts_by_key',
    'build_garch_trade_range_history_context',
    'garch_cone_components_by_window',
    'garch_default_components',
):
    globals().pop(_garch_output_name, None)

garch_option_error = None
try:

    # Refresh the shared plotter so notebook reruns pick up local code edits without a kernel restart.
    from importlib import reload
    import Quantapp.visualization.line_chart_plotter as _line_chart_plotter_module
    reload(_line_chart_plotter_module)
    lineChartPlotter = _line_chart_plotter_module.LineChartPlotter()

    from scipy.stats import norm

    try:
        from arch import arch_model
    except ModuleNotFoundError as exc:
        if exc.name != 'arch':
            raise
        raise ImportError(
            "The GARCH option requires the 'arch' package. Install it in the notebook kernel environment with `pip install arch`."
        ) from exc

    trade_range_garch_window = trade_range_default_window if 'trade_range_default_window' in globals() else 50
    trade_range_garch_window_options = trade_range_window_options if 'trade_range_window_options' in globals() else [trade_range_garch_window]
    trade_range_garch_interval_levels = trade_range_interval_levels if 'trade_range_interval_levels' in globals() else [0.95, 0.99]
    trade_range_garch_tail_levels = trade_range_tail_levels if 'trade_range_tail_levels' in globals() else [0.95, 0.99]

    # Fit the model to completed close-to-close returns so the forecast stays aligned with the simplified trade-range view.
    garch_trade_frame = trade_range_price_frame.dropna().copy()
    garch_session_returns = garch_trade_frame['Close'].pct_change().dropna()
    if len(garch_session_returns) < 3:
        raise ValueError(
            'The GARCH option needs at least three sessions with valid close prices to fit a GARCH trade-range variant.'
        )

    garch_completed_frame = garch_trade_frame.iloc[:-1].copy()
    garch_completed_returns = garch_trade_frame['Close'].pct_change().iloc[:-1].dropna()
    if len(garch_completed_returns) < 50:
        raise ValueError(
            f'The GARCH option needs at least 50 completed sessions to fit a stable GARCH trade-range variant. Only {len(garch_completed_returns)} are available.'
        )

    # Historical diagnostics use a filtered full-sample GARCH fit across completed sessions.
    garch_history_model = arch_model(
        garch_completed_returns.mul(100.0),
        mean='Constant',
        vol='GARCH',
        p=1,
        q=1,
        dist='normal',
        rescale=False,
    )
    garch_history_fit = garch_history_model.fit(disp='off')
    garch_history_mean_return = float(garch_history_fit.params.get('mu', 0.0) / 100.0)
    garch_history_sigma_series = pd.Series(
        garch_history_fit.conditional_volatility / 100.0,
        index=garch_completed_returns.index,
    )
    garch_history_open = garch_trade_frame['Close'].shift(1).astype(float).reindex(garch_completed_returns.index)
    garch_history_close = garch_completed_frame['Close'].astype(float).reindex(garch_completed_returns.index)
    garch_current_history_forecast = garch_history_fit.forecast(horizon=1, reindex=False)
    garch_current_history_mean_return = float(garch_current_history_forecast.mean.iloc[-1, 0] / 100.0)
    garch_current_history_sigma_return = float(np.sqrt(garch_current_history_forecast.variance.iloc[-1, 0]) / 100.0)
    if not np.isfinite(garch_current_history_sigma_return) or garch_current_history_sigma_return <= 0:
        raise ValueError(
            'The GARCH option could not produce a positive current-session GARCH volatility forecast for the historical diagnostics.'
        )
    garch_current_history_date = garch_trade_frame.index[-1]
    garch_current_history_return = float((garch_trade_frame.attrs.get('current_session_latest_price', garch_trade_frame['Close'].iloc[-1]) / risk_distribution_analytics._resolve_current_reference_price(garch_trade_frame)) - 1.0)
    garch_history_metrics_by_confidence = {}
    for confidence in sorted(set(trade_range_garch_interval_levels).union(trade_range_garch_tail_levels)):
        alpha = 1.0 - confidence
        interval_alpha = (1.0 - confidence) / 2.0
        interval_z = float(norm.ppf(interval_alpha))
        z_alpha = float(norm.ppf(alpha))
        pdf_alpha = float(norm.pdf(z_alpha))

        lower_interval_return = garch_history_mean_return + (garch_history_sigma_series * interval_z)
        upper_interval_return = garch_history_mean_return - (garch_history_sigma_series * interval_z)
        lower_var_return = garch_history_mean_return + (garch_history_sigma_series * z_alpha)
        upper_var_return = garch_history_mean_return - (garch_history_sigma_series * z_alpha)
        lower_expected_shortfall_return = garch_history_mean_return - (garch_history_sigma_series * (pdf_alpha / alpha))
        upper_expected_shortfall_return = garch_history_mean_return + (garch_history_sigma_series * (pdf_alpha / alpha))

        current_lower_var_return = garch_current_history_mean_return + (garch_current_history_sigma_return * z_alpha)
        current_upper_var_return = garch_current_history_mean_return - (garch_current_history_sigma_return * z_alpha)
        current_lower_expected_shortfall_return = (
            garch_current_history_mean_return - (garch_current_history_sigma_return * (pdf_alpha / alpha))
        )
        current_upper_expected_shortfall_return = (
            garch_current_history_mean_return + (garch_current_history_sigma_return * (pdf_alpha / alpha))
        )

        lower_var_return_with_current = pd.concat([
            lower_var_return,
            pd.Series([current_lower_var_return], index=[garch_current_history_date]),
        ]).sort_index()
        upper_var_return_with_current = pd.concat([
            upper_var_return,
            pd.Series([current_upper_var_return], index=[garch_current_history_date]),
        ]).sort_index()
        lower_expected_shortfall_return_with_current = pd.concat([
            lower_expected_shortfall_return,
            pd.Series([current_lower_expected_shortfall_return], index=[garch_current_history_date]),
        ]).sort_index()
        upper_expected_shortfall_return_with_current = pd.concat([
            upper_expected_shortfall_return,
            pd.Series([current_upper_expected_shortfall_return], index=[garch_current_history_date]),
        ]).sort_index()

        lower_breaches = garch_completed_returns.lt(lower_var_return).astype(float)
        upper_breaches = garch_completed_returns.gt(upper_var_return).astype(float)
        lower_breaches_with_current = pd.concat([
            lower_breaches,
            pd.Series([float(garch_current_history_return < current_lower_var_return)], index=[garch_current_history_date]),
        ]).sort_index()
        upper_breaches_with_current = pd.concat([
            upper_breaches,
            pd.Series([float(garch_current_history_return > current_upper_var_return)], index=[garch_current_history_date]),
        ]).sort_index()
        either_side_breaches_with_current = lower_breaches_with_current.add(upper_breaches_with_current, fill_value=0.0).clip(upper=1.0)
        lower_rolling_breach_rate = lower_breaches_with_current.rolling(trade_range_garch_window).mean().dropna()
        upper_rolling_breach_rate = upper_breaches_with_current.rolling(trade_range_garch_window).mean().dropna()
        either_side_rolling_breach_rate = either_side_breaches_with_current.rolling(trade_range_garch_window).mean().dropna()

        lower_expected_breach_rate = pd.Series(
            data=np.full(len(lower_rolling_breach_rate.index), alpha, dtype=float),
            index=lower_rolling_breach_rate.index,
        )
        upper_expected_breach_rate = pd.Series(
            data=np.full(len(upper_rolling_breach_rate.index), alpha, dtype=float),
            index=upper_rolling_breach_rate.index,
        )
        either_side_expected_breach_rate = pd.Series(
            data=np.full(len(either_side_rolling_breach_rate.index), min(1.0, 2.0 * alpha), dtype=float),
            index=either_side_rolling_breach_rate.index,
        )

        open_for_interval = garch_history_open.reindex(lower_interval_return.index)
        open_for_var = garch_history_open.reindex(lower_var_return.index)
        open_for_es = garch_history_open.reindex(lower_expected_shortfall_return.index)

        garch_history_metrics_by_confidence[confidence] = {
            'session_returns': garch_completed_returns,
            'session_open': garch_history_open,
            'session_close': garch_history_close,
            'lower_interval_return': lower_interval_return,
            'upper_interval_return': upper_interval_return,
            'lower_interval_price': open_for_interval.mul(1.0 + lower_interval_return),
            'upper_interval_price': open_for_interval.mul(1.0 + upper_interval_return),
            'lower_var_return': lower_var_return_with_current,
            'upper_var_return': upper_var_return_with_current,
            'lower_var_price': open_for_var.mul(1.0 + lower_var_return),
            'upper_var_price': open_for_var.mul(1.0 + upper_var_return),
            'lower_expected_shortfall_return': lower_expected_shortfall_return_with_current,
            'upper_expected_shortfall_return': upper_expected_shortfall_return_with_current,
            'lower_expected_shortfall_price': open_for_es.mul(1.0 + lower_expected_shortfall_return),
            'upper_expected_shortfall_price': open_for_es.mul(1.0 + upper_expected_shortfall_return),
            'lower_breaches': lower_breaches_with_current,
            'upper_breaches': upper_breaches_with_current,
            'either_side_breaches': either_side_breaches_with_current,
            'lower_rolling_breach_rate': lower_rolling_breach_rate,
            'upper_rolling_breach_rate': upper_rolling_breach_rate,
            'either_side_rolling_breach_rate': either_side_rolling_breach_rate,
            'lower_expected_breach_rate': lower_expected_breach_rate,
            'upper_expected_breach_rate': upper_expected_breach_rate,
            'either_side_expected_breach_rate': either_side_expected_breach_rate,
        }

    garch_trade_history_context = {
        'window': trade_range_garch_window,
        'horizon_sessions': trade_range_default_horizon,
        'interval_confidence_levels': trade_range_garch_interval_levels,
        'tail_confidence_levels': trade_range_garch_tail_levels,
        'metrics_by_confidence': garch_history_metrics_by_confidence,
        'session_returns': garch_session_returns,
        'session_open': garch_trade_frame['Close'].shift(1).astype(float),
        'session_close': garch_trade_frame['Close'].astype(float),
    }
    garch_trade_history_fig = lineChartPlotter.plot_trade_range_history_profile(
        history_context=garch_trade_history_context,
        ticker_label=f'{ticker_str} GARCH(1,1) Normal',
    )
    garch_trade_history_contexts_by_key = {
        (int(trade_range_garch_window), int(trade_range_default_horizon)): garch_trade_history_context,
    }

    def build_garch_trade_range_history_context(window, horizon_sessions=1):
        window = int(window)
        horizon_sessions = int(horizon_sessions)
        history_cache_key = (window, horizon_sessions)
        cached_history_context = garch_trade_history_contexts_by_key.get(history_cache_key)
        if cached_history_context is not None:
            return cached_history_context

        if horizon_sessions == int(trade_range_default_horizon):
            history_session_returns = garch_session_returns.copy()
            history_session_open = garch_trade_frame['Close'].shift(1).astype(float).reindex(history_session_returns.index)
            history_session_close = garch_trade_frame['Close'].astype(float).reindex(history_session_returns.index)
            history_fit_returns = garch_completed_returns.copy()
            include_current_session = True
        else:
            history_holding_frame = risk_distribution_analytics._build_session_holding_period_frame(
                garch_trade_frame,
                horizon_sessions,
            )
            history_session_returns = history_holding_frame['session_return'].dropna()
            history_session_open = history_holding_frame['session_open'].astype(float).reindex(history_session_returns.index)
            history_session_close = history_holding_frame['session_close'].astype(float).reindex(history_session_returns.index)
            history_fit_returns = history_session_returns.copy()
            include_current_session = False

        if len(history_fit_returns) < 50:
            raise ValueError(
                f'The GARCH option needs at least 50 completed {horizon_sessions}-session returns to build the selected history view. Only {len(history_fit_returns)} are available.'
            )

        history_model = arch_model(
            history_fit_returns.mul(100.0),
            mean='Constant',
            vol='GARCH',
            p=1,
            q=1,
            dist='normal',
            rescale=False,
        )
        history_fit = history_model.fit(disp='off')
        history_mean_return = float(history_fit.params.get('mu', 0.0) / 100.0)
        history_sigma_series = pd.Series(
            history_fit.conditional_volatility / 100.0,
            index=history_fit_returns.index,
        )

        current_history_date = None
        current_history_return = None
        current_history_mean_return = None
        current_history_sigma_return = None
        if include_current_session:
            current_history_forecast = history_fit.forecast(horizon=1, reindex=False)
            current_history_mean_return = float(current_history_forecast.mean.iloc[-1, 0] / 100.0)
            current_history_sigma_return = float(np.sqrt(current_history_forecast.variance.iloc[-1, 0]) / 100.0)
            if not np.isfinite(current_history_sigma_return) or current_history_sigma_return <= 0:
                raise ValueError(
                    'The GARCH option could not produce a positive current-session GARCH volatility forecast for the selected history view.'
                )
            current_history_date = garch_trade_frame.index[-1]
            current_history_return = float(garch_session_returns.iloc[-1])

        history_metrics_by_confidence = {}
        for confidence in sorted(set(trade_range_garch_interval_levels).union(trade_range_garch_tail_levels)):
            alpha = 1.0 - confidence
            interval_alpha = (1.0 - confidence) / 2.0
            interval_z = float(norm.ppf(interval_alpha))
            z_alpha = float(norm.ppf(alpha))
            pdf_alpha = float(norm.pdf(z_alpha))

            lower_interval_return = history_mean_return + (history_sigma_series * interval_z)
            upper_interval_return = history_mean_return - (history_sigma_series * interval_z)
            lower_var_return = history_mean_return + (history_sigma_series * z_alpha)
            upper_var_return = history_mean_return - (history_sigma_series * z_alpha)
            lower_expected_shortfall_return = history_mean_return - (history_sigma_series * (pdf_alpha / alpha))
            upper_expected_shortfall_return = history_mean_return + (history_sigma_series * (pdf_alpha / alpha))

            if include_current_session:
                current_lower_var_return = current_history_mean_return + (current_history_sigma_return * z_alpha)
                current_upper_var_return = current_history_mean_return - (current_history_sigma_return * z_alpha)
                current_lower_expected_shortfall_return = (
                    current_history_mean_return - (current_history_sigma_return * (pdf_alpha / alpha))
                )
                current_upper_expected_shortfall_return = (
                    current_history_mean_return + (current_history_sigma_return * (pdf_alpha / alpha))
                )

                lower_var_return_for_plot = pd.concat([
                    lower_var_return,
                    pd.Series([current_lower_var_return], index=[current_history_date]),
                ]).sort_index()
                upper_var_return_for_plot = pd.concat([
                    upper_var_return,
                    pd.Series([current_upper_var_return], index=[current_history_date]),
                ]).sort_index()
                lower_expected_shortfall_return_for_plot = pd.concat([
                    lower_expected_shortfall_return,
                    pd.Series([current_lower_expected_shortfall_return], index=[current_history_date]),
                ]).sort_index()
                upper_expected_shortfall_return_for_plot = pd.concat([
                    upper_expected_shortfall_return,
                    pd.Series([current_upper_expected_shortfall_return], index=[current_history_date]),
                ]).sort_index()

                lower_breaches = history_fit_returns.lt(lower_var_return).astype(float)
                upper_breaches = history_fit_returns.gt(upper_var_return).astype(float)
                lower_breaches_for_plot = pd.concat([
                    lower_breaches,
                    pd.Series([float(current_history_return < current_lower_var_return)], index=[current_history_date]),
                ]).sort_index()
                upper_breaches_for_plot = pd.concat([
                    upper_breaches,
                    pd.Series([float(current_history_return > current_upper_var_return)], index=[current_history_date]),
                ]).sort_index()
            else:
                lower_var_return_for_plot = lower_var_return.copy()
                upper_var_return_for_plot = upper_var_return.copy()
                lower_expected_shortfall_return_for_plot = lower_expected_shortfall_return.copy()
                upper_expected_shortfall_return_for_plot = upper_expected_shortfall_return.copy()
                lower_breaches_for_plot = history_session_returns.lt(lower_var_return_for_plot).astype(float)
                upper_breaches_for_plot = history_session_returns.gt(upper_var_return_for_plot).astype(float)

            either_side_breaches_for_plot = lower_breaches_for_plot.add(upper_breaches_for_plot, fill_value=0.0).clip(upper=1.0)
            lower_rolling_breach_rate = lower_breaches_for_plot.rolling(window).mean().dropna()
            upper_rolling_breach_rate = upper_breaches_for_plot.rolling(window).mean().dropna()
            either_side_rolling_breach_rate = either_side_breaches_for_plot.rolling(window).mean().dropna()

            lower_expected_breach_rate = pd.Series(
                data=np.full(len(lower_rolling_breach_rate.index), alpha, dtype=float),
                index=lower_rolling_breach_rate.index,
            )
            upper_expected_breach_rate = pd.Series(
                data=np.full(len(upper_rolling_breach_rate.index), alpha, dtype=float),
                index=upper_rolling_breach_rate.index,
            )
            either_side_expected_breach_rate = pd.Series(
                data=np.full(len(either_side_rolling_breach_rate.index), min(1.0, 2.0 * alpha), dtype=float),
                index=either_side_rolling_breach_rate.index,
            )

            open_for_interval = history_session_open.reindex(lower_interval_return.index)
            open_for_var = history_session_open.reindex(lower_var_return_for_plot.index)
            open_for_es = history_session_open.reindex(lower_expected_shortfall_return_for_plot.index)

            history_metrics_by_confidence[confidence] = {
                'session_returns': history_session_returns,
                'session_open': history_session_open,
                'session_close': history_session_close,
                'lower_interval_return': lower_interval_return,
                'upper_interval_return': upper_interval_return,
                'lower_interval_price': open_for_interval.mul(1.0 + lower_interval_return),
                'upper_interval_price': open_for_interval.mul(1.0 + upper_interval_return),
                'lower_var_return': lower_var_return_for_plot,
                'upper_var_return': upper_var_return_for_plot,
                'lower_var_price': open_for_var.mul(1.0 + lower_var_return_for_plot),
                'upper_var_price': open_for_var.mul(1.0 + upper_var_return_for_plot),
                'lower_expected_shortfall_return': lower_expected_shortfall_return_for_plot,
                'upper_expected_shortfall_return': upper_expected_shortfall_return_for_plot,
                'lower_expected_shortfall_price': open_for_es.mul(1.0 + lower_expected_shortfall_return_for_plot),
                'upper_expected_shortfall_price': open_for_es.mul(1.0 + upper_expected_shortfall_return_for_plot),
                'lower_breaches': lower_breaches_for_plot,
                'upper_breaches': upper_breaches_for_plot,
                'either_side_breaches': either_side_breaches_for_plot,
                'lower_rolling_breach_rate': lower_rolling_breach_rate,
                'upper_rolling_breach_rate': upper_rolling_breach_rate,
                'either_side_rolling_breach_rate': either_side_rolling_breach_rate,
                'lower_expected_breach_rate': lower_expected_breach_rate,
                'upper_expected_breach_rate': upper_expected_breach_rate,
                'either_side_expected_breach_rate': either_side_expected_breach_rate,
            }

        history_context = {
            'window': window,
            'horizon_sessions': horizon_sessions,
            'interval_confidence_levels': trade_range_garch_interval_levels,
            'tail_confidence_levels': trade_range_garch_tail_levels,
            'metrics_by_confidence': history_metrics_by_confidence,
            'session_returns': history_session_returns,
            'session_open': history_session_open,
            'session_close': history_session_close,
        }
        garch_trade_history_contexts_by_key[history_cache_key] = history_context
        return history_context
    from plotly.subplots import make_subplots
    import copy as _copy


    def _axis_index_from_ref(axis_ref, axis_letter):
        if axis_ref in (None, axis_letter):
            return 1
        if isinstance(axis_ref, str) and axis_ref.startswith(axis_letter):
            digits = ''.join(ch for ch in axis_ref[1:] if ch.isdigit())
            return int(digits) if digits else 1
        return 1


    def _shift_axis_ref(axis_ref, row_offset):
        if not isinstance(axis_ref, str) or axis_ref == 'paper':
            return axis_ref
        suffix = ' domain' if axis_ref.endswith(' domain') else ''
        base = axis_ref[:-7] if suffix else axis_ref
        if not base or base[0] not in ('x', 'y'):
            return axis_ref
        axis_letter = base[0]
        digits = base[1:]
        axis_index = int(digits) if digits else 1
        shifted_index = axis_index + row_offset
        shifted_base = axis_letter if shifted_index == 1 else f'{axis_letter}{shifted_index}'
        return shifted_base + suffix


    def _stack_trade_range_figures(history_fig, cone_fig, *, title_text):
        return _compose_trade_range_stack(
            history_fig,
            cone_fig,
            title_text=title_text,
        )


    # Current-session forecast uses the selected trailing fit window so the cone can switch windows like Block 7.
    def build_garch_trade_range_cone_components(window, horizon_sessions=1):
        window = int(window)
        horizon_sessions = max(1, int(horizon_sessions))

        garch_fit_returns = garch_completed_returns.tail(window).dropna()
        garch_fit_window = len(garch_fit_returns)
        if garch_fit_window < 21:
            raise ValueError(
                f'The GARCH option needs at least 21 completed sessions inside the trailing fit window. Only {garch_fit_window} are available for the {window}-session view.'
            )

        garch_trade_model = arch_model(
            garch_fit_returns.mul(100.0),
            mean='Constant',
            vol='GARCH',
            p=1,
            q=1,
            dist='normal',
            rescale=False,
        )
        garch_trade_fit = garch_trade_model.fit(disp='off')
        garch_trade_forecast = garch_trade_fit.forecast(horizon=horizon_sessions, reindex=False)

        forecast_mean_values = np.asarray(garch_trade_forecast.mean.iloc[-1], dtype=float)
        forecast_variance_values = np.asarray(garch_trade_forecast.variance.iloc[-1], dtype=float)
        if len(forecast_mean_values) < horizon_sessions or len(forecast_variance_values) < horizon_sessions:
            raise ValueError(
                f'The GARCH option could not produce a complete {horizon_sessions}-session forecast for the {window}-session fit window.'
            )

        garch_mean_return = float(np.nansum(forecast_mean_values[:horizon_sessions]) / 100.0)
        garch_sigma_return = float(
            np.sqrt(np.clip(forecast_variance_values[:horizon_sessions], a_min=0.0, a_max=None).sum()) / 100.0
        )
        if not np.isfinite(garch_sigma_return) or garch_sigma_return <= 0:
            raise ValueError(
                f'The GARCH option could not produce a positive {horizon_sessions}-session volatility forecast for the {window}-session fit window.'
            )

        garch_holding_frame = risk_distribution_analytics._build_session_holding_period_frame(
            garch_trade_frame,
            horizon_sessions,
        )
        garch_sample_returns = garch_holding_frame['session_return'].iloc[:-1].tail(window).dropna()
        garch_effective_window = len(garch_sample_returns)
        if garch_effective_window == 0:
            raise ValueError(
                f'The GARCH option could not build a completed {horizon_sessions}-session return sample for the {window}-session fit window.'
            )

        garch_anchor_price = float(risk_distribution_analytics._resolve_current_reference_price(garch_trade_frame))
        garch_latest_price = float(garch_trade_frame.attrs.get('current_session_latest_price', garch_trade_frame['Close'].iloc[-1]))
        garch_session_date = pd.Timestamp(garch_trade_frame.attrs.get('current_session_date', garch_trade_frame.index[-1]))
        garch_median_return = garch_mean_return
        garch_median_price = garch_anchor_price * (1.0 + garch_median_return)

        garch_model_summary = pd.DataFrame([
            {
                'Model': 'GARCH(1,1) Normal',
                'Fit Window': garch_fit_window,
                'Forecast Horizon': horizon_sessions,
                'Forecast Mean Return': garch_mean_return,
                'Forecast Sigma Return': garch_sigma_return,
                'Omega': float(garch_trade_fit.params.get('omega', np.nan)),
                'Alpha(1)': float(garch_trade_fit.params.get('alpha[1]', np.nan)),
                'Beta(1)': float(garch_trade_fit.params.get('beta[1]', np.nan)),
                'Alpha + Beta': float(garch_trade_fit.params.get('alpha[1]', np.nan) + garch_trade_fit.params.get('beta[1]', np.nan)),
            }
        ])

        garch_interval_map = {}
        garch_range_rows = []
        for confidence in trade_range_garch_interval_levels:
            tail_probability = (1.0 - confidence) / 2.0
            lower_return = float(norm.ppf(tail_probability, loc=garch_mean_return, scale=garch_sigma_return))
            upper_return = float(norm.ppf(1.0 - tail_probability, loc=garch_mean_return, scale=garch_sigma_return))
            lower_price = garch_anchor_price * (1.0 + lower_return)
            upper_price = garch_anchor_price * (1.0 + upper_return)
            garch_interval_map[confidence] = {
                'lower_return': lower_return,
                'upper_return': upper_return,
                'lower_price': lower_price,
                'upper_price': upper_price,
            }
            garch_range_rows.append(
                {
                    'Confidence': f'{confidence:.0%}',
                    'Lower Return': lower_return,
                    'Upper Return': upper_return,
                    'Lower Exit Price': lower_price,
                    'Upper Exit Price': upper_price,
                }
            )

        garch_long_tail_map = {}
        garch_short_tail_map = {}
        garch_tail_rows = []
        for confidence in trade_range_garch_tail_levels:
            alpha = 1.0 - confidence
            z_alpha = float(norm.ppf(alpha))
            pdf_alpha = float(norm.pdf(z_alpha))

            long_var_return = garch_mean_return + (garch_sigma_return * z_alpha)
            long_cvar_return = garch_mean_return - (garch_sigma_return * (pdf_alpha / alpha))
            short_var_return = garch_mean_return - (garch_sigma_return * z_alpha)
            short_cvar_return = garch_mean_return + (garch_sigma_return * (pdf_alpha / alpha))

            garch_long_tail_map[confidence] = {
                'var_return': long_var_return,
                'var_price': garch_anchor_price * (1.0 + long_var_return),
                'expected_shortfall_return': long_cvar_return,
                'expected_shortfall_price': garch_anchor_price * (1.0 + long_cvar_return),
            }
            garch_short_tail_map[confidence] = {
                'var_return': short_var_return,
                'var_price': garch_anchor_price * (1.0 + short_var_return),
                'expected_shortfall_return': short_cvar_return,
                'expected_shortfall_price': garch_anchor_price * (1.0 + short_cvar_return),
            }
            garch_tail_rows.append(
                {
                    'Confidence': f'{confidence:.0%}',
                    'Long VaR Floor': garch_long_tail_map[confidence]['var_price'],
                    'Long CVaR Floor': garch_long_tail_map[confidence]['expected_shortfall_price'],
                    'Short VaR Ceiling': garch_short_tail_map[confidence]['var_price'],
                    'Short CVaR Ceiling': garch_short_tail_map[confidence]['expected_shortfall_price'],
                }
            )

        garch_range_summary = pd.DataFrame(garch_range_rows)
        garch_tail_summary = pd.DataFrame(garch_tail_rows).rename(columns={
            'Confidence': 'Tail Confidence',
        })

        garch_trade_range_context = {
            'session_date': garch_session_date,
            'window': window,
            'horizon_sessions': horizon_sessions,
            'effective_window': garch_effective_window,
            'anchor_price': garch_anchor_price,
            'latest_price': garch_latest_price,
            'sample_returns': garch_sample_returns,
            'interval_confidence_levels': trade_range_garch_interval_levels,
            'tail_confidence_levels': trade_range_garch_tail_levels,
            'intervals': garch_interval_map,
            'long_tail_levels': garch_long_tail_map,
            'short_tail_levels': garch_short_tail_map,
            'median_return': garch_median_return,
            'median_price': garch_median_price,
        }

        return {
            'context': garch_trade_range_context,
            'model_summary': garch_model_summary,
            'range_summary': garch_range_summary,
            'tail_summary': garch_tail_summary,
        }

    garch_cone_components_by_key = {
        (int(trade_range_garch_window), int(trade_range_default_horizon)): build_garch_trade_range_cone_components(
            int(trade_range_garch_window),
            int(trade_range_default_horizon),
        )
    }
    garch_default_components = garch_cone_components_by_key[
        (int(trade_range_garch_window), int(trade_range_default_horizon))
    ]

    formatted_garch_model_summary = garch_default_components['model_summary'].copy()
    for column in ('Forecast Mean Return', 'Forecast Sigma Return'):
        if column in formatted_garch_model_summary.columns:
            formatted_garch_model_summary[column] = formatted_garch_model_summary[column].map(
                lambda value: f'{value:.2%}' if pd.notna(value) else value
            )
    for column in ('Omega', 'Alpha(1)', 'Beta(1)', 'Alpha + Beta'):
        if column in formatted_garch_model_summary.columns:
            formatted_garch_model_summary[column] = formatted_garch_model_summary[column].map(
                lambda value: f'{value:.4f}' if pd.notna(value) else value
            )

    garch_range_summary = garch_default_components['range_summary'].copy()
    if not garch_range_summary.empty:
        for column in ('Lower Return', 'Upper Return'):
            if column in garch_range_summary.columns:
                garch_range_summary[column] = garch_range_summary[column].map(
                    lambda value: f'{value:.2%}' if pd.notna(value) else value
                )
        for column in ('Lower Exit Price', 'Upper Exit Price'):
            if column in garch_range_summary.columns:
                garch_range_summary[column] = garch_range_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

    garch_tail_summary = garch_default_components['tail_summary'].copy()
    if not garch_tail_summary.empty:
        for column in (
            'Long VaR Floor',
            'Long CVaR Floor',
            'Short VaR Ceiling',
            'Short CVaR Ceiling',
        ):
            if column in garch_tail_summary.columns:
                garch_tail_summary[column] = garch_tail_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

except Exception as exc:
    garch_option_error = exc
    print(f'GARCH option unavailable: {exc}')

import plotly.graph_objects as go


def _trade_range_horizon_label(horizon):
    horizon = int(horizon)
    return '1-session' if horizon == 1 else f'{horizon}-session'


def _trade_range_history_basis(horizon):
    horizon = int(horizon)
    if horizon == 1:
        return 'completed close-to-close return history'
    return f'completed {horizon}-session close-based holding-period returns'


def _trade_range_confidence_label(confidence):
    return f'{float(confidence):.0%}'


def _trade_range_confidence_key(levels, confidence):
    target = float(confidence)
    for level in levels:
        if np.isclose(float(level), target):
            return level
    raise KeyError(f'Trade-range confidence {target:.0%} is not available.')


def _coerce_trade_range_confidence(confidence):
    if confidence in (None, ''):
        return float(trade_range_default_confidence)
    return float(_trade_range_confidence_key(trade_range_confidence_options, confidence))


def _filter_trade_range_table_by_confidence(frame, confidence, column_name=None):
    if frame is None:
        return frame
    filtered_frame = frame.copy()
    if filtered_frame.empty:
        return filtered_frame
    selected_label = _trade_range_confidence_label(confidence)
    candidate_columns = [column_name] if column_name is not None else ['Confidence', 'Tail Confidence']
    for candidate_column in candidate_columns:
        if candidate_column in filtered_frame.columns:
            return filtered_frame.loc[
                filtered_frame[candidate_column].eq(selected_label)
            ].reset_index(drop=True)
    return filtered_frame


def _filter_trade_range_history_context(history_context, confidence):
    selected_confidence = _trade_range_confidence_key(
        history_context['metrics_by_confidence'].keys(),
        confidence,
    )
    filtered_history_context = dict(history_context)
    filtered_history_context['interval_confidence_levels'] = [float(selected_confidence)]
    filtered_history_context['tail_confidence_levels'] = [float(selected_confidence)]
    filtered_history_context['metrics_by_confidence'] = {
        selected_confidence: history_context['metrics_by_confidence'][selected_confidence]
    }
    return filtered_history_context


def _filter_trade_range_cone_context(cone_context, confidence):
    selected_confidence = _trade_range_confidence_key(cone_context['intervals'].keys(), confidence)
    filtered_cone_context = dict(cone_context)
    filtered_cone_context['interval_confidence_levels'] = [float(selected_confidence)]
    filtered_cone_context['tail_confidence_levels'] = [float(selected_confidence)]
    filtered_cone_context['intervals'] = {
        selected_confidence: cone_context['intervals'][selected_confidence]
    }
    filtered_cone_context['long_tail_levels'] = {
        selected_confidence: cone_context['long_tail_levels'][selected_confidence]
    }
    filtered_cone_context['short_tail_levels'] = {
        selected_confidence: cone_context['short_tail_levels'][selected_confidence]
    }
    if 'range_summary_table' in cone_context:
        filtered_cone_context['range_summary_table'] = _filter_trade_range_table_by_confidence(
            cone_context['range_summary_table'],
            selected_confidence,
            'Confidence',
        )
    if 'tail_summary_table' in cone_context:
        filtered_cone_context['tail_summary_table'] = _filter_trade_range_table_by_confidence(
            cone_context['tail_summary_table'],
            selected_confidence,
            'Confidence',
        )
    return filtered_cone_context


def _filter_garch_trade_range_components(components, confidence):
    filtered_context = _filter_trade_range_cone_context(components['context'], confidence)
    selected_confidence = filtered_context['interval_confidence_levels'][0]
    return {
        'context': filtered_context,
        'model_summary': components['model_summary'].copy(),
        'range_summary': _filter_trade_range_table_by_confidence(
            components['range_summary'],
            selected_confidence,
            'Confidence',
        ),
        'tail_summary': _filter_trade_range_table_by_confidence(
            components['tail_summary'],
            selected_confidence,
            'Tail Confidence',
        ),
    }


def _format_empirical_trade_range_tables(cone_context):
    formatted_range_summary = cone_context['range_summary_table'].copy().rename(columns={
        'Lower Close': 'Lower Exit Price',
        'Upper Close': 'Upper Exit Price',
    })
    formatted_tail_summary = cone_context['tail_summary_table'].copy().rename(columns={
        'Confidence': 'Tail Confidence',
    })

    if not formatted_range_summary.empty:
        for column in ('Lower Return', 'Upper Return'):
            if column in formatted_range_summary.columns:
                formatted_range_summary[column] = formatted_range_summary[column].map(
                    lambda value: f'{value:.2%}' if pd.notna(value) else value
                )
        for column in ('Lower Exit Price', 'Upper Exit Price'):
            if column in formatted_range_summary.columns:
                formatted_range_summary[column] = formatted_range_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

    if not formatted_tail_summary.empty:
        for column in (
            'Long VaR Floor',
            'Long CVaR Floor',
            'Short VaR Ceiling',
            'Short CVaR Ceiling',
        ):
            if column in formatted_tail_summary.columns:
                formatted_tail_summary[column] = formatted_tail_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

    return formatted_range_summary, formatted_tail_summary


def _format_garch_trade_range_tables(components):
    formatted_model_summary = components['model_summary'].copy()
    for column in ('Forecast Mean Return', 'Forecast Sigma Return'):
        if column in formatted_model_summary.columns:
            formatted_model_summary[column] = formatted_model_summary[column].map(
                lambda value: f'{value:.2%}' if pd.notna(value) else value
            )
    for column in ('Omega', 'Alpha(1)', 'Beta(1)', 'Alpha + Beta'):
        if column in formatted_model_summary.columns:
            formatted_model_summary[column] = formatted_model_summary[column].map(
                lambda value: f'{value:.4f}' if pd.notna(value) else value
            )

    formatted_range_summary = components['range_summary'].copy()
    if not formatted_range_summary.empty:
        for column in ('Lower Return', 'Upper Return'):
            if column in formatted_range_summary.columns:
                formatted_range_summary[column] = formatted_range_summary[column].map(
                    lambda value: f'{value:.2%}' if pd.notna(value) else value
                )
        for column in ('Lower Exit Price', 'Upper Exit Price'):
            if column in formatted_range_summary.columns:
                formatted_range_summary[column] = formatted_range_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

    formatted_tail_summary = components['tail_summary'].copy()
    if not formatted_tail_summary.empty:
        for column in (
            'Long VaR Floor',
            'Long CVaR Floor',
            'Short VaR Ceiling',
            'Short CVaR Ceiling',
        ):
            if column in formatted_tail_summary.columns:
                formatted_tail_summary[column] = formatted_tail_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

    return formatted_model_summary, formatted_range_summary, formatted_tail_summary


def _strip_trade_range_dropdowns(fig):
    stripped_fig = _copy.deepcopy(fig)
    stripped_fig.update_layout(updatemenus=[])
    return stripped_fig


def _build_empirical_trade_range_view(window, horizon, confidence=None):
    window = int(window)
    horizon = int(horizon)
    confidence = _coerce_trade_range_confidence(confidence)
    cached_history_context = trade_range_history_contexts_by_horizon.get(horizon)
    if cached_history_context is not None and window in cached_history_context['metrics_by_window']:
        history_context = {
            'window': window,
            'horizon_sessions': horizon,
            'interval_confidence_levels': cached_history_context['interval_confidence_levels'],
            'tail_confidence_levels': cached_history_context['tail_confidence_levels'],
            'metrics_by_confidence': cached_history_context['metrics_by_window'][window],
            'session_returns': cached_history_context['session_returns'],
            'session_open': cached_history_context['session_open'],
            'session_close': cached_history_context['session_close'],
        }
    else:
        history_context = risk_distribution_analytics.build_trade_range_history_context(
            price_frame=trade_range_price_frame,
            windows=[window],
            horizon_sessions=horizon,
            interval_confidence_levels=trade_range_interval_levels,
            tail_confidence_levels=trade_range_tail_levels,
            default_window=window,
        )
        history_context = {
            'window': window,
            'horizon_sessions': horizon,
            'interval_confidence_levels': history_context['interval_confidence_levels'],
            'tail_confidence_levels': history_context['tail_confidence_levels'],
            'metrics_by_confidence': history_context['metrics_by_confidence'],
            'session_returns': history_context['session_returns'],
            'session_open': history_context['session_open'],
            'session_close': history_context['session_close'],
        }
    history_context = _filter_trade_range_history_context(history_context, confidence)
    history_fig = _strip_trade_range_dropdowns(
        lineChartPlotter.plot_trade_range_history_profile(
            history_context=history_context,
            ticker_label=ticker_str,
        )
    )
    cone_cache_key = (window, horizon)
    cone_context = trade_range_cone_contexts_by_key.get(cone_cache_key)
    if cone_context is None:
        cone_context = risk_distribution_analytics.build_trade_range_probability_context(
            price_frame=trade_range_price_frame,
            window=window,
            horizon_sessions=horizon,
            interval_confidence_levels=trade_range_interval_levels,
            tail_confidence_levels=trade_range_tail_levels,
        )
        trade_range_cone_contexts_by_key[cone_cache_key] = cone_context
    filtered_cone_context = _filter_trade_range_cone_context(cone_context, confidence)
    range_summary_window, tail_summary_window = _format_empirical_trade_range_tables(filtered_cone_context)
    cone_fig = _strip_trade_range_dropdowns(
        lineChartPlotter.plot_trade_range_probability_cone(
            cone_context=filtered_cone_context,
            ticker_label=ticker_str,
        )
    )
    combined_fig = _stack_trade_range_figures(
        history_fig,
        cone_fig,
        title_text=(
            f'{ticker_str} Historical Two-Sided Trade Range Profile and Current Cone '
            f'({int(window)}-Session Lookback, {int(horizon)}-Session Horizon)'
        ),
    )
    combined_fig.update_layout(updatemenus=[])
    return {
        'caption': (
            f'Empirical trade range using {_trade_range_history_basis(horizon)} '
            f'at {_trade_range_confidence_label(confidence)} confidence.'
        ),
        'model_summary': None,
        'range_summary': range_summary_window,
        'tail_summary': tail_summary_window,
        'cone_figure': cone_fig,
        'figure': combined_fig,
    }


def _build_garch_trade_range_view(window, horizon, confidence=None):
    window = int(window)
    horizon = int(horizon)
    confidence = _coerce_trade_range_confidence(confidence)
    garch_cache_key = (window, horizon)
    if garch_cache_key not in garch_cone_components_by_key:
        garch_cone_components_by_key[garch_cache_key] = build_garch_trade_range_cone_components(window, horizon)
    components = _filter_garch_trade_range_components(
        garch_cone_components_by_key[garch_cache_key],
        confidence,
    )
    formatted_model_summary_window, range_summary_window, tail_summary_window = _format_garch_trade_range_tables(components)
    history_context = _filter_trade_range_history_context(
        build_garch_trade_range_history_context(window, horizon),
        confidence,
    )
    history_fig = _strip_trade_range_dropdowns(
        lineChartPlotter.plot_trade_range_history_profile(
            history_context=history_context,
            ticker_label=f'{ticker_str} GARCH(1,1) Normal',
        )
    )
    cone_fig = _strip_trade_range_dropdowns(
        lineChartPlotter.plot_trade_range_probability_cone(
            cone_context=components['context'],
            ticker_label=f'{ticker_str} GARCH(1,1)',
        )
    )
    combined_fig = _stack_trade_range_figures(
        history_fig,
        cone_fig,
        title_text=(
            f'{ticker_str} GARCH(1,1) Historical Diagnostics and Current Cone '
            f'({int(window)}-Session Fit Window, {int(horizon)}-Session Horizon)'
        ),
    )
    combined_fig.update_layout(updatemenus=[])
    if horizon == 1:
        caption = (
            f'GARCH(1,1)-normal trade range fit on completed one-session returns at '
            f'{_trade_range_confidence_label(confidence)} confidence. '
            'Session lookback changes the current-session cone, and the historical diagnostics stay on the same one-session basis.'
        )
    else:
        caption = (
            f'GARCH(1,1)-normal view filtered to {_trade_range_confidence_label(confidence)} confidence. '
            f'GARCH(1,1)-normal current cones still aggregate one-session forecasts across the selected {horizon}-session horizon. '
            f'The historical diagnostics now plot completed {horizon}-session holding returns through a horizon-matched GARCH filter so the rolling return panel stays aligned with the selected horizon.'
        )
    return {
        'caption': caption,
        'model_summary': formatted_model_summary_window,
        'range_summary': range_summary_window,
        'tail_summary': tail_summary_window,
        'cone_figure': cone_fig,
        'figure': combined_fig,
    }


trade_range_view_builders = {
    'Empirical': _build_empirical_trade_range_view,
}
trade_range_window_options_by_method = {
    'Empirical': list(trade_range_window_options),
}

if all(name in globals() for name in ('garch_trade_history_fig', 'garch_cone_components_by_key')):
    trade_range_view_builders['GARCH(1,1) Normal'] = _build_garch_trade_range_view
    trade_range_window_options_by_method['GARCH(1,1) Normal'] = list(trade_range_garch_window_options)

trade_range_view_cache = {}


def get_trade_range_view(method, window, horizon, confidence=None):
    confidence = _coerce_trade_range_confidence(confidence)
    cache_key = (str(method), int(window), int(horizon), float(confidence))
    if cache_key not in trade_range_view_cache:
        trade_range_view_cache[cache_key] = trade_range_view_builders[str(method)](
            int(window),
            int(horizon),
            float(confidence),
        )
    return trade_range_view_cache[cache_key]


def _set_default_trade_range_outputs():
    default_empirical_view = get_trade_range_view(
        'Empirical',
        int(trade_range_default_window),
        int(trade_range_default_horizon),
        float(trade_range_default_confidence),
    )
    globals()['range_summary'] = default_empirical_view['range_summary'].copy()
    globals()['tail_summary'] = default_empirical_view['tail_summary'].copy()
    globals()['trade_range_fig'] = default_empirical_view['cone_figure']
    globals()['trade_range_combined_fig'] = default_empirical_view['figure']

    if 'GARCH(1,1) Normal' in trade_range_view_builders:
        default_garch_window = trade_range_garch_window if 'trade_range_garch_window' in globals() else trade_range_window_options_by_method['GARCH(1,1) Normal'][0]
        default_garch_view = get_trade_range_view(
            'GARCH(1,1) Normal',
            int(default_garch_window),
            int(trade_range_default_horizon),
            float(trade_range_default_confidence),
        )
        globals()['formatted_garch_model_summary'] = default_garch_view['model_summary'].copy()
        globals()['garch_range_summary'] = default_garch_view['range_summary'].copy()
        globals()['garch_tail_summary'] = default_garch_view['tail_summary'].copy()
        globals()['garch_trade_range_fig'] = default_garch_view['cone_figure']
        globals()['garch_trade_combined_fig'] = default_garch_view['figure']


from dash import Dash, Input, Output, State, dash_table, dcc, html
import socket
import uuid

required_trade_range_globals = (
    'get_trade_range_view',
    'trade_range_window_options_by_method',
    'trade_range_default_window',
    'trade_range_default_horizon',
)
missing_trade_range_globals = [name for name in required_trade_range_globals if name not in globals()]
if missing_trade_range_globals:
    raise NameError(
        'Run Block 7 before Block 7B. Missing globals: ' + ', '.join(missing_trade_range_globals)
    )



_set_default_trade_range_outputs()

def _dash_trade_range_window_bounds(method):
    method = str(method)
    min_window = 21
    if method == 'GARCH(1,1) Normal' and 'garch_completed_returns' in globals():
        max_window = int(len(garch_completed_returns))
    else:
        max_window = int(len(trade_range_price_frame.dropna()) - 1)
    max_window = max(min_window, max_window)
    return min_window, max_window


def _dash_trade_range_preferred_window(method):
    method = str(method)
    preferred_window = int(trade_range_default_window)
    if method == 'GARCH(1,1) Normal' and 'trade_range_garch_window' in globals():
        preferred_window = int(trade_range_garch_window)
    min_window, max_window = _dash_trade_range_window_bounds(method)
    return max(min_window, min(max_window, preferred_window))


def _coerce_dash_trade_range_window(method, value):
    min_window, max_window = _dash_trade_range_window_bounds(method)
    preferred_window = _dash_trade_range_preferred_window(method)
    helper_text = f'Available range: {min_window} to {max_window} completed sessions.'

    if value in (None, ''):
        return preferred_window, min_window, max_window, helper_text

    try:
        normalized_value = int(value)
    except (TypeError, ValueError):
        return (
            preferred_window,
            min_window,
            max_window,
            f'{helper_text} Using {preferred_window}-session lookback because the input was not a whole number.',
        )

    if normalized_value < min_window:
        return (
            min_window,
            min_window,
            max_window,
            f'{helper_text} Using {min_window}-session lookback because the requested value was below the supported minimum.',
        )

    if normalized_value > max_window:
        return (
            max_window,
            min_window,
            max_window,
            f'{helper_text} Using {max_window}-session lookback because the request exceeded the available completed-session history.',
        )

    return normalized_value, min_window, max_window, helper_text


def _dash_trade_range_default_window(method, current_window=None):
    if current_window is None:
        return _dash_trade_range_preferred_window(method)
    return _coerce_dash_trade_range_window(method, current_window)[0]


def _dash_trade_range_horizon_bounds(method, window):
    window, _, _, _ = _coerce_dash_trade_range_window(method, window)
    available_sessions = int(len(trade_range_price_frame.dropna()))
    max_horizon = max(1, (available_sessions - int(window) + 1) // 2)
    return 1, max_horizon


def _dash_trade_range_preferred_horizon(method, window):
    min_horizon, max_horizon = _dash_trade_range_horizon_bounds(method, window)
    preferred_horizon = int(trade_range_default_horizon)
    return max(min_horizon, min(max_horizon, preferred_horizon))


def _coerce_dash_trade_range_horizon(method, window, value):
    min_horizon, max_horizon = _dash_trade_range_horizon_bounds(method, window)
    preferred_horizon = _dash_trade_range_preferred_horizon(method, window)
    helper_text = (
        f'Available range: {min_horizon} to {max_horizon} sessions. '
        '1 session means close-to-close.'
    )

    if value in (None, ''):
        return preferred_horizon, min_horizon, max_horizon, helper_text

    try:
        normalized_value = int(value)
    except (TypeError, ValueError):
        return (
            preferred_horizon,
            min_horizon,
            max_horizon,
            f'{helper_text} Using {preferred_horizon}-session horizon because the input was not a whole number.',
        )

    if normalized_value < min_horizon:
        return (
            min_horizon,
            min_horizon,
            max_horizon,
            f'{helper_text} Using {min_horizon}-session horizon because the requested value was below the supported minimum.',
        )

    if normalized_value > max_horizon:
        return (
            max_horizon,
            min_horizon,
            max_horizon,
            f'{helper_text} Using {max_horizon}-session horizon because the request exceeded the available completed-session history.',
        )

    return normalized_value, min_horizon, max_horizon, helper_text


def _dash_trade_range_default_horizon(method, window, current_horizon=None):
    if current_horizon is None:
        return _dash_trade_range_preferred_horizon(method, window)
    return _coerce_dash_trade_range_horizon(method, window, current_horizon)[0]


def _dash_trade_range_snapshot_table(method, window, horizon, confidence, view):
    rows = [
        {'Metric': 'Method', 'Value': method},
        {'Metric': 'Session Lookback Window', 'Value': f'{int(window)}-session'},
        {'Metric': 'Holding Horizon', 'Value': f'{int(horizon)}-session'},
        {'Metric': 'Confidence Level', 'Value': _trade_range_confidence_label(confidence)},
        {'Metric': 'Description', 'Value': view['caption']},
    ]

    if method == 'GARCH(1,1) Normal' and view['model_summary'] is not None and not view['model_summary'].empty:
        summary_row = view['model_summary'].iloc[0]
        preferred_fields = [
            ('Forecast Mean', 'Forecast Mean Return'),
            ('Forecast Sigma', 'Forecast Sigma Return'),
            ('Omega', 'Omega'),
            ('Alpha(1)', 'Alpha(1)'),
            ('Beta(1)', 'Beta(1)'),
            ('Alpha + Beta', 'Alpha + Beta'),
        ]
        for label, column in preferred_fields:
            if column in summary_row.index:
                rows.append({'Metric': label, 'Value': str(summary_row[column])})
    else:
        rows.append({'Metric': 'Data Basis', 'Value': _trade_range_history_basis(horizon)})

    return pd.DataFrame(rows)


def _dash_table_payload(frame):
    if frame is None or frame.empty:
        frame = pd.DataFrame({'Info': ['No data available']})
    frame = frame.fillna('')
    columns = [{'name': column, 'id': column} for column in frame.columns]
    data = frame.astype(str).to_dict('records')
    return columns, data


def _find_open_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(('127.0.0.1', 0))
        sock.listen(1)
        return sock.getsockname()[1]


dash_trade_range_id = f"trade-range-dash-{uuid.uuid4().hex[:8]}"
dash_trade_range_methods = [str(method) for method in trade_range_window_options_by_method.keys()]
dash_trade_range_default_method = 'Empirical' if 'Empirical' in dash_trade_range_methods else dash_trade_range_methods[0]
dash_trade_range_default_window = _dash_trade_range_default_window(dash_trade_range_default_method)
dash_trade_range_default_horizon = _dash_trade_range_default_horizon(
    dash_trade_range_default_method,
    dash_trade_range_default_window,
)
dash_trade_range_default_confidence = float(trade_range_default_confidence)

dash_trade_range_app = Dash(dash_trade_range_id)
dash_trade_range_app.layout = html.Div(
    [
        html.Div(
            [
                html.Div(
                    [
                        html.Label('Method', style={'display': 'block', 'marginBottom': '6px', 'fontWeight': '600'}),
                        dcc.Dropdown(
                            id=f'{dash_trade_range_id}-method',
                            options=[{'label': method, 'value': method} for method in dash_trade_range_methods],
                            value=dash_trade_range_default_method,
                            clearable=False,
                        ),
                    ],
                    style={'flex': '1 1 320px', 'minWidth': '260px'},
                ),
                html.Div(
                    [
                        html.Label('Session Lookback Window', style={'display': 'block', 'marginBottom': '6px', 'fontWeight': '600'}),
                        dcc.Input(
                            id=f'{dash_trade_range_id}-window',
                            type='number',
                            value=dash_trade_range_default_window,
                            min=_dash_trade_range_window_bounds(dash_trade_range_default_method)[0],
                            max=_dash_trade_range_window_bounds(dash_trade_range_default_method)[1],
                            step=1,
                            debounce=True,
                            inputMode='numeric',
                            style={'width': '100%'},
                        ),
                        html.Div(
                            id=f'{dash_trade_range_id}-window-status',
                            style={'marginTop': '6px', 'fontSize': '12px', 'color': '#94a3b8'},
                        ),
                    ],
                    style={'flex': '0 0 220px', 'minWidth': '180px'},
                ),
                html.Div(
                    [
                        html.Label('Holding Horizon (Sessions)', style={'display': 'block', 'marginBottom': '6px', 'fontWeight': '600'}),
                        dcc.Input(
                            id=f'{dash_trade_range_id}-horizon',
                            type='number',
                            value=dash_trade_range_default_horizon,
                            min=_dash_trade_range_horizon_bounds(
                                dash_trade_range_default_method,
                                dash_trade_range_default_window,
                            )[0],
                            max=_dash_trade_range_horizon_bounds(
                                dash_trade_range_default_method,
                                dash_trade_range_default_window,
                            )[1],
                            step=1,
                            debounce=True,
                            inputMode='numeric',
                            style={'width': '100%'},
                        ),
                        html.Div(
                            id=f'{dash_trade_range_id}-horizon-status',
                            style={'marginTop': '6px', 'fontSize': '12px', 'color': '#94a3b8'},
                        ),
                    ],
                    style={'flex': '0 0 220px', 'minWidth': '180px'},
                ),
                html.Div(
                    [
                        html.Label('Confidence', style={'display': 'block', 'marginBottom': '6px', 'fontWeight': '600'}),
                        dcc.Dropdown(
                            id=f'{dash_trade_range_id}-confidence',
                            options=[
                                {
                                    'label': _trade_range_confidence_label(confidence),
                                    'value': float(confidence),
                                }
                                for confidence in trade_range_confidence_options
                            ],
                            value=dash_trade_range_default_confidence,
                            clearable=False,
                        ),
                    ],
                    style={'flex': '0 0 200px', 'minWidth': '180px'},
                ),
            ],
            style={'display': 'flex', 'gap': '12px', 'flexWrap': 'wrap', 'marginBottom': '14px'},
        ),
        html.Div(id=f'{dash_trade_range_id}-caption', style={'marginBottom': '14px', 'fontWeight': '600'}),
        html.Div(
            [
                html.Div('Model Snapshot', style={'marginBottom': '8px', 'fontWeight': '600'}),
                dash_table.DataTable(
                    id=f'{dash_trade_range_id}-snapshot-table',
                    style_table={'overflowX': 'auto', 'width': '100%'},
                    style_cell={
                        'textAlign': 'left',
                        'padding': '6px 10px',
                        'backgroundColor': '#0f172a',
                        'color': '#e2e8f0',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                        'whiteSpace': 'normal',
                        'height': 'auto',
                    },
                    style_header={
                        'backgroundColor': '#1e293b',
                        'color': '#f8fafc',
                        'fontWeight': '700',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                    },
                ),
            ],
            style={'marginBottom': '18px'},
        ),
        html.Div(
            [
                html.Div('Projected Return / Price Range Summary', style={'marginBottom': '8px', 'fontWeight': '600'}),
                dash_table.DataTable(
                    id=f'{dash_trade_range_id}-range-table',
                    style_table={'overflowX': 'auto', 'width': '100%'},
                    style_cell={
                        'textAlign': 'left',
                        'padding': '6px 10px',
                        'backgroundColor': '#0f172a',
                        'color': '#e2e8f0',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                        'whiteSpace': 'normal',
                        'height': 'auto',
                    },
                    style_header={
                        'backgroundColor': '#1e293b',
                        'color': '#f8fafc',
                        'fontWeight': '700',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                    },
                ),
            ],
            style={'marginBottom': '18px'},
        ),
        html.Div(
            [
                html.Div('Projected Tail Summary', style={'marginBottom': '8px', 'fontWeight': '600'}),
                dash_table.DataTable(
                    id=f'{dash_trade_range_id}-tail-table',
                    style_table={'overflowX': 'auto', 'width': '100%'},
                    style_cell={
                        'textAlign': 'left',
                        'padding': '6px 10px',
                        'backgroundColor': '#0f172a',
                        'color': '#e2e8f0',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                        'whiteSpace': 'normal',
                        'height': 'auto',
                    },
                    style_header={
                        'backgroundColor': '#1e293b',
                        'color': '#f8fafc',
                        'fontWeight': '700',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                    },
                ),
            ],
            style={'marginBottom': '18px'},
        ),
        dcc.Graph(
            id=f'{dash_trade_range_id}-figure',
            style={'width': '100%', 'height': '3000px'},
            config={'responsive': True, 'displaylogo': False},
        ),
    ],
    style={
        'width': '100%',
        'maxWidth': '100%',
        'padding': '4px 0 16px 0',
        'fontFamily': 'Inter, Segoe UI, sans-serif',
    },
)


@dash_trade_range_app.callback(
    Output(f'{dash_trade_range_id}-window', 'value'),
    Output(f'{dash_trade_range_id}-window', 'min'),
    Output(f'{dash_trade_range_id}-window', 'max'),
    Input(f'{dash_trade_range_id}-method', 'value'),
    State(f'{dash_trade_range_id}-window', 'value'),
)
def _sync_dash_trade_range_window(method, current_window):
    selected_window, min_window, max_window, _ = _coerce_dash_trade_range_window(method, current_window)
    return selected_window, min_window, max_window


@dash_trade_range_app.callback(
    Output(f'{dash_trade_range_id}-horizon', 'value'),
    Output(f'{dash_trade_range_id}-horizon', 'min'),
    Output(f'{dash_trade_range_id}-horizon', 'max'),
    Input(f'{dash_trade_range_id}-method', 'value'),
    Input(f'{dash_trade_range_id}-window', 'value'),
    State(f'{dash_trade_range_id}-horizon', 'value'),
)
def _sync_dash_trade_range_horizon(method, window, current_horizon):
    selected_window, _, _, _ = _coerce_dash_trade_range_window(method, window)
    selected_horizon, min_horizon, max_horizon, _ = _coerce_dash_trade_range_horizon(
        method,
        selected_window,
        current_horizon,
    )
    return selected_horizon, min_horizon, max_horizon


@dash_trade_range_app.callback(
    Output(f'{dash_trade_range_id}-caption', 'children'),
    Output(f'{dash_trade_range_id}-window-status', 'children'),
    Output(f'{dash_trade_range_id}-horizon-status', 'children'),
    Output(f'{dash_trade_range_id}-snapshot-table', 'columns'),
    Output(f'{dash_trade_range_id}-snapshot-table', 'data'),
    Output(f'{dash_trade_range_id}-range-table', 'columns'),
    Output(f'{dash_trade_range_id}-range-table', 'data'),
    Output(f'{dash_trade_range_id}-tail-table', 'columns'),
    Output(f'{dash_trade_range_id}-tail-table', 'data'),
    Output(f'{dash_trade_range_id}-figure', 'figure'),
    Input(f'{dash_trade_range_id}-method', 'value'),
    Input(f'{dash_trade_range_id}-window', 'value'),
    Input(f'{dash_trade_range_id}-horizon', 'value'),
    Input(f'{dash_trade_range_id}-confidence', 'value'),
)
def _render_dash_trade_range_view(method, window, horizon, confidence):
    method = str(method)
    window, _, _, window_status = _coerce_dash_trade_range_window(method, window)
    horizon, _, _, horizon_status = _coerce_dash_trade_range_horizon(method, window, horizon)
    confidence = _coerce_trade_range_confidence(confidence)
    view = get_trade_range_view(method, window, horizon, confidence)

    snapshot_columns, snapshot_data = _dash_table_payload(
        _dash_trade_range_snapshot_table(method, window, horizon, confidence, view)
    )
    range_columns, range_data = _dash_table_payload(view['range_summary'])
    tail_columns, tail_data = _dash_table_payload(view['tail_summary'])

    figure = _copy.deepcopy(view['figure'])
    figure.update_layout(autosize=True)

    caption = (
        f'{method} | {window}-session lookback | {horizon}-session horizon | '
        f'{_trade_range_confidence_label(confidence)} confidence | {view["caption"]}'
    )
    return (
        caption,
        window_status,
        horizon_status,
        snapshot_columns,
        snapshot_data,
        range_columns,
        range_data,
        tail_columns,
        tail_data,
        figure,
    )


dash_trade_range_port = _find_open_port()
print(f'Dash Block 7B preview starting on port {dash_trade_range_port}.')
dash_trade_range_app.run(
    port=dash_trade_range_port,
    debug=False,
    jupyter_mode='inline',
    jupyter_width='100%',
    jupyter_height=3900,
    dev_tools_ui=False,
    dev_tools_props_check=False,
)


In [ ]:
# Block 7C: calibrate empirical breach rates across holding horizons and lookback windows

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from numpy.lib.stride_tricks import sliding_window_view
from plotly.subplots import make_subplots

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


trade_range_breach_windows = [21, 50, 200]
trade_range_breach_confidence_levels = list(globals().get('trade_range_tail_levels', [0.95, 0.99]))
trade_range_breach_price_frame = globals().get(
    'trade_range_price_frame',
    globals().get('ticker_trade_range_source', ticker)[['Open', 'Close']].copy(),
)
trade_range_breach_complete_frame = (
    trade_range_breach_price_frame[['Open', 'Close']].dropna().sort_index().copy()
)
trade_range_breach_complete_rows = int(len(trade_range_breach_complete_frame))
if trade_range_breach_complete_rows < 2:
    raise ValueError('Block 7C requires at least two completed sessions with Open and Close prices.')

trade_range_breach_target_end = min(21, trade_range_breach_complete_rows)
trade_range_breach_horizon_step = 1
trade_range_breach_confidence_labels = {
    float(confidence): f'{confidence:.0%}' for confidence in trade_range_breach_confidence_levels
}
trade_range_breach_window_labels = [f'{int(window)}-session' for window in trade_range_breach_windows]
trade_range_breach_horizon_grid = sorted(
    {
        1,
        *range(
            min(trade_range_breach_horizon_step, trade_range_breach_target_end),
            trade_range_breach_target_end + 1,
            trade_range_breach_horizon_step,
        ),
        trade_range_breach_target_end,
    }
)
trade_range_breach_tick_step = 20 if trade_range_breach_target_end > 120 else 10
trade_range_breach_tick_values = [
    horizon
    for horizon in trade_range_breach_horizon_grid
    if horizon in (1, trade_range_breach_target_end) or horizon % trade_range_breach_tick_step == 0
]
trade_range_breach_cache_signature = (
    tuple(int(window) for window in trade_range_breach_windows),
    tuple(float(confidence) for confidence in trade_range_breach_confidence_levels),
    int(trade_range_breach_complete_rows),
    str(trade_range_breach_complete_frame.index[0]),
    str(trade_range_breach_complete_frame.index[-1]),
    float(trade_range_breach_complete_frame['Open'].iloc[0]),
    float(trade_range_breach_complete_frame['Open'].iloc[-1]),
    float(trade_range_breach_complete_frame['Close'].iloc[0]),
    float(trade_range_breach_complete_frame['Close'].iloc[-1]),
)
if globals().get('trade_range_breach_cache_signature') != trade_range_breach_cache_signature:
    trade_range_breach_snapshot_cache = {}
else:
    trade_range_breach_snapshot_cache = globals().get('trade_range_breach_snapshot_cache', {})
globals()['trade_range_breach_cache_signature'] = trade_range_breach_cache_signature


def _build_trade_range_breach_horizon_snapshot(horizon):
    horizon = int(horizon)
    if horizon <= 0:
        raise ValueError('horizon must be a positive integer.')
    if horizon > trade_range_breach_complete_rows:
        raise ValueError('horizon exceeds the available completed session count.')

    cached_snapshot = trade_range_breach_snapshot_cache.get(horizon)
    if isinstance(cached_snapshot, pd.DataFrame) and not cached_snapshot.empty:
        return cached_snapshot.copy()

    holding_frame = risk_distribution_analytics._build_session_holding_period_frame(
        trade_range_breach_complete_frame,
        horizon,
    )
    session_returns = holding_frame['session_return'].to_numpy(dtype=float)
    latest_calibration_date = pd.Timestamp(holding_frame.index[-1])

    records = []
    for window in trade_range_breach_windows:
        rolling_window = int(window)
        if len(session_returns) < rolling_window:
            rolling_samples = None
            valid_count = 0
            realized_returns = np.array([], dtype=float)
        else:
            rolling_samples = sliding_window_view(session_returns, rolling_window)
            valid_count = len(session_returns) - horizon - rolling_window + 1
            realized_returns = session_returns[horizon + rolling_window - 1:]

        for confidence in trade_range_breach_confidence_levels:
            alpha = 1.0 - float(confidence)
            actual_breach_rate = np.nan
            expected_breach_rate = np.nan
            calibration_date = pd.NaT

            if rolling_samples is not None and valid_count > 0:
                lower_var = np.quantile(rolling_samples, alpha, axis=1)[:valid_count]
                upper_var = np.quantile(rolling_samples, 1.0 - alpha, axis=1)[:valid_count]
                breaches = np.logical_or(
                    realized_returns < lower_var,
                    realized_returns > upper_var,
                ).astype(float)
                if len(breaches) >= rolling_window:
                    actual_breach_rate = float(breaches[-rolling_window:].mean())
                    expected_breach_rate = float(min(1.0, 2.0 * alpha))
                    calibration_date = latest_calibration_date

            records.append(
                {
                    'horizon': horizon,
                    'window': rolling_window,
                    'confidence': float(confidence),
                    'latest_calibration_date': calibration_date,
                    'actual_breach_rate': actual_breach_rate,
                    'expected_breach_rate': expected_breach_rate,
                    'excess_breach_rate': (
                        actual_breach_rate - expected_breach_rate
                        if pd.notna(actual_breach_rate) and pd.notna(expected_breach_rate)
                        else np.nan
                    ),
                }
            )

    snapshot_frame = pd.DataFrame(records)
    trade_range_breach_snapshot_cache[horizon] = snapshot_frame
    globals()['trade_range_breach_snapshot_cache'] = trade_range_breach_snapshot_cache
    return snapshot_frame.copy()


def _build_trade_range_breach_summary():
    summary_frames = [
        _build_trade_range_breach_horizon_snapshot(horizon)
        for horizon in trade_range_breach_horizon_grid
    ]
    summary = pd.concat(summary_frames, ignore_index=True).sort_values(
        ['confidence', 'window', 'horizon']
    ).reset_index(drop=True)

    support_text = (
        f'Displaying holding horizons 1 through {trade_range_breach_target_end} '
        f'in single-session steps across lookbacks {", ".join(str(window) for window in trade_range_breach_windows)}. '
        'Blank cells mean there was not enough completed history to measure the latest rolling breach rate '
        'for that horizon and lookback combination.'
    )
    return summary, list(trade_range_breach_horizon_grid), support_text


def _build_trade_range_breach_heatmap_payload(summary_frame, confidence, supported_horizons):
    confidence_frame = summary_frame.loc[
        summary_frame['confidence'].eq(float(confidence))
    ].copy()
    confidence_frame['date_label'] = confidence_frame['latest_calibration_date'].map(
        lambda value: pd.Timestamp(value).strftime('%Y-%m-%d') if pd.notna(value) else ''
    )

    excess_rows = []
    customdata_rows = []
    for window in trade_range_breach_windows:
        window_frame = confidence_frame.loc[
            confidence_frame['window'].eq(int(window))
        ].set_index('horizon').reindex(supported_horizons)

        excess_row = window_frame['excess_breach_rate'].mul(100.0).tolist()
        actual_row = window_frame['actual_breach_rate'].mul(100.0).tolist()
        expected_row = window_frame['expected_breach_rate'].mul(100.0).tolist()
        date_row = window_frame['date_label'].tolist()

        excess_rows.append(excess_row)
        customdata_rows.append(
            [
                [actual, expected, excess, date_label]
                for actual, expected, excess, date_label in zip(
                    actual_row,
                    expected_row,
                    excess_row,
                    date_row,
                )
            ]
        )

    return np.asarray(excess_rows, dtype=float), np.asarray(customdata_rows, dtype=object)


def _build_trade_range_breach_excess_figure(summary_frame, supported_horizons):
    excess_values = summary_frame['excess_breach_rate'].dropna()
    color_limit = (
        float(excess_values.abs().max() * 100.0)
        if not excess_values.empty
        else 1.0
    )
    color_limit = max(color_limit, 1.0)

    fig = make_subplots(
        rows=len(trade_range_breach_confidence_levels),
        cols=1,
        subplot_titles=tuple(
            f'{trade_range_breach_confidence_labels[float(confidence)]} Either-Side Excess Breach Rate'
            for confidence in trade_range_breach_confidence_levels
        ),
        vertical_spacing=0.12,
    )

    for confidence_index, confidence in enumerate(trade_range_breach_confidence_levels, start=1):
        z_values, customdata_values = _build_trade_range_breach_heatmap_payload(
            summary_frame,
            confidence,
            supported_horizons,
        )
        fig.add_trace(
            go.Heatmap(
                z=z_values,
                x=supported_horizons,
                y=trade_range_breach_window_labels,
                coloraxis='coloraxis',
                customdata=customdata_values,
                xgap=1,
                ygap=1,
                hovertemplate=(
                    'Holding Horizon: %{x}-session'
                    '<br>Session Lookback: %{y}'
                    f'<br>Confidence: {trade_range_breach_confidence_labels[float(confidence)]}'
                    '<br>Actual Either-Side Breach Rate: %{customdata[0]:.2f}%'
                    '<br>Expected Either-Side Breach Rate: %{customdata[1]:.2f}%'
                    '<br>Excess Either-Side Breach Rate: %{customdata[2]:.2f} pts'
                    '<br>Latest Calibration Date: %{customdata[3]}'
                    '<extra></extra>'
                ),
            ),
            row=confidence_index,
            col=1,
        )

        fig.update_xaxes(
            title_text='Holding Horizon (Sessions)',
            tickmode='array',
            tickvals=trade_range_breach_tick_values,
            row=confidence_index,
            col=1,
        )
        fig.update_yaxes(
            title_text='Session Lookback Window',
            categoryorder='array',
            categoryarray=trade_range_breach_window_labels,
            autorange='reversed',
            row=confidence_index,
            col=1,
        )

    fig.update_layout(
        title=(
            f'{ticker_str} Empirical Either-Side Breach-Rate Calibration '
            f'(Holding Horizons 1-{trade_range_breach_target_end} Sessions; '
            f'Lookbacks {" / ".join(str(window) for window in trade_range_breach_windows)})'
        ),
        template='plotly_dark',
        height=980,
        margin={'t': 120, 'r': 40, 'b': 70, 'l': 90},
        coloraxis={
            'colorscale': 'RdBu',
            'cmin': -color_limit,
            'cmax': color_limit,
            'cmid': 0.0,
            'colorbar': {'title': 'Excess Breach Rate (pts)'},
        },
    )
    return fig


def _format_trade_range_breach_summary(summary_frame):
    formatted_summary = summary_frame.copy()
    formatted_summary['Holding Horizon'] = formatted_summary['horizon'].map(
        lambda value: f'{int(value)}-session'
    )
    formatted_summary['Session Lookback Window'] = formatted_summary['window'].map(
        lambda value: f'{int(value)}-session'
    )
    formatted_summary['Confidence'] = formatted_summary['confidence'].map(
        lambda value: f'{value:.0%}'
    )
    formatted_summary['Latest Calibration Date'] = formatted_summary['latest_calibration_date'].map(
        lambda value: pd.Timestamp(value).strftime('%Y-%m-%d') if pd.notna(value) else ''
    )
    for column in ('actual_breach_rate', 'expected_breach_rate', 'excess_breach_rate'):
        formatted_summary[column] = formatted_summary[column].map(
            lambda value: f'{value:.2%}' if pd.notna(value) else ''
        )
    return formatted_summary.rename(
        columns={
            'actual_breach_rate': 'Actual Either-Side Breach Rate',
            'expected_breach_rate': 'Expected Either-Side Breach Rate',
            'excess_breach_rate': 'Excess Either-Side Breach Rate',
        }
    )[
        [
            'Holding Horizon',
            'Session Lookback Window',
            'Confidence',
            'Latest Calibration Date',
            'Actual Either-Side Breach Rate',
            'Expected Either-Side Breach Rate',
            'Excess Either-Side Breach Rate',
        ]
    ]


trade_range_breach_calibration_summary, trade_range_breach_supported_horizons, trade_range_breach_support_text = (
    _build_trade_range_breach_summary()
)
trade_range_breach_calibration_summary_display = _format_trade_range_breach_summary(
    trade_range_breach_calibration_summary
)
trade_range_breach_excess_fig = _build_trade_range_breach_excess_figure(
    trade_range_breach_calibration_summary,
    trade_range_breach_supported_horizons,
)

globals()['trade_range_breach_supported_horizons'] = list(trade_range_breach_supported_horizons)
globals()['trade_range_breach_calibration_summary'] = trade_range_breach_calibration_summary.copy()
globals()['trade_range_breach_calibration_summary_display'] = (
    trade_range_breach_calibration_summary_display.copy()
)
globals()['trade_range_breach_excess_fig'] = trade_range_breach_excess_fig

print(trade_range_breach_support_text)
display(trade_range_breach_excess_fig)
display(trade_range_breach_calibration_summary_display)


In [ ]:
# Block 7D: plot average empirical breach rates across all holding horizons over time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if 'trade_range_breach_calibration_summary' not in globals():
    raise ValueError('Run Block 7C before Block 7D to build the breach-rate calibration summary.')

trade_range_breach_average_price_frame = globals().get(
    'trade_range_breach_price_frame',
    globals().get('trade_range_price_frame', globals().get('ticker_trade_range_source', ticker)[['Open', 'Close']].copy()),
)
trade_range_breach_average_price_frame = (
    trade_range_breach_average_price_frame[['Open', 'Close']].dropna().sort_index().copy()
)
trade_range_breach_average_horizons = [
    int(horizon)
    for horizon in globals().get(
        'trade_range_breach_supported_horizons',
        sorted(int(value) for value in globals()['trade_range_breach_calibration_summary']['horizon'].dropna().unique()),
    )
]
trade_range_breach_average_windows = [
    int(window)
    for window in globals().get(
        'trade_range_breach_windows',
        sorted(int(value) for value in globals()['trade_range_breach_calibration_summary']['window'].dropna().unique()),
    )
]
trade_range_breach_average_confidences = [
    float(confidence)
    for confidence in globals().get(
        'trade_range_breach_confidence_levels',
        sorted(float(value) for value in globals()['trade_range_breach_calibration_summary']['confidence'].dropna().unique()),
    )
]
trade_range_breach_average_signature = (
    tuple(trade_range_breach_average_horizons),
    tuple(trade_range_breach_average_windows),
    tuple(trade_range_breach_average_confidences),
    int(len(trade_range_breach_average_price_frame)),
    str(trade_range_breach_average_price_frame.index[0]),
    str(trade_range_breach_average_price_frame.index[-1]),
    float(trade_range_breach_average_price_frame['Open'].iloc[0]),
    float(trade_range_breach_average_price_frame['Open'].iloc[-1]),
    float(trade_range_breach_average_price_frame['Close'].iloc[0]),
    float(trade_range_breach_average_price_frame['Close'].iloc[-1]),
)
if globals().get('trade_range_breach_average_signature') != trade_range_breach_average_signature:
    trade_range_breach_average_contexts_by_horizon = {}
else:
    trade_range_breach_average_contexts_by_horizon = globals().get(
        'trade_range_breach_average_contexts_by_horizon',
        {},
    )
globals()['trade_range_breach_average_signature'] = trade_range_breach_average_signature

trade_range_breach_average_palette = ['#38bdf8', '#22c55e', '#f59e0b', '#f97316', '#a855f7']
trade_range_breach_average_colors = {
    int(window): trade_range_breach_average_palette[index % len(trade_range_breach_average_palette)]
    for index, window in enumerate(trade_range_breach_average_windows)
}


def _build_trade_range_breach_average_context(horizon):
    horizon = int(horizon)
    cached_context = trade_range_breach_average_contexts_by_horizon.get(horizon)
    if isinstance(cached_context, dict) and cached_context:
        return cached_context

    history_context = risk_distribution_analytics.build_trade_range_history_context(
        price_frame=trade_range_breach_average_price_frame,
        windows=trade_range_breach_average_windows,
        horizon_sessions=horizon,
        interval_confidence_levels=trade_range_breach_average_confidences,
        tail_confidence_levels=trade_range_breach_average_confidences,
        default_window=trade_range_breach_average_windows[0],
    )
    trade_range_breach_average_contexts_by_horizon[horizon] = history_context
    globals()['trade_range_breach_average_contexts_by_horizon'] = trade_range_breach_average_contexts_by_horizon
    return history_context


def _build_trade_range_breach_average_panel():
    averaged_series = {}
    horizon_count_series = {}
    support_lines = []

    for confidence in trade_range_breach_average_confidences:
        for window in trade_range_breach_average_windows:
            horizon_series = []
            for horizon in trade_range_breach_average_horizons:
                history_context = _build_trade_range_breach_average_context(horizon)
                metric_set = history_context['metrics_by_window'][int(window)][float(confidence)]
                breach_rate_series = metric_set.get('either_side_rolling_breach_rate', pd.Series(dtype=float)).dropna()
                if breach_rate_series.empty:
                    continue
                horizon_series.append(
                    breach_rate_series.rename(f'h{int(horizon)}')
                )

            if not horizon_series:
                continue

            aligned_frame = pd.concat(horizon_series, axis=1, join='inner').dropna(how='any')
            if aligned_frame.empty:
                continue

            averaged_series[(float(confidence), int(window))] = aligned_frame.mean(axis=1)
            horizon_count_series[(float(confidence), int(window))] = pd.Series(
                data=np.full(len(aligned_frame.index), aligned_frame.shape[1], dtype=int),
                index=aligned_frame.index,
            )
            support_lines.append(
                f'{float(confidence):.0%} / {int(window)}-session starts {aligned_frame.index.min():%Y-%m-%d} '
                f'with all {aligned_frame.shape[1]} horizons aligned.'
            )

    if not averaged_series:
        raise ValueError('Unable to build any averaged breach-rate series across the configured horizons.')

    average_panel = pd.concat(averaged_series, axis=1).sort_index(axis=1)
    count_panel = pd.concat(horizon_count_series, axis=1).sort_index(axis=1)
    support_text = 'Averages are taken across all configured holding horizons on dates where every horizon-specific rolling breach-rate series is available. '
    support_text += ' '.join(sorted(set(support_lines)))
    return average_panel, count_panel, support_text


def _build_trade_range_breach_average_figure(average_panel, count_panel):
    subplot_titles = tuple(
        f'{float(confidence):.0%} Average Either-Side Rolling Breach Rate Across Horizons'
        for confidence in trade_range_breach_average_confidences
    )
    fig = make_subplots(
        rows=len(trade_range_breach_average_confidences),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.10 if len(trade_range_breach_average_confidences) > 1 else 0.12,
        subplot_titles=subplot_titles,
    )

    for row_index, confidence in enumerate(trade_range_breach_average_confidences, start=1):
        expected_rate = float(min(1.0, 2.0 * (1.0 - float(confidence))))
        for window in trade_range_breach_average_windows:
            column_key = (float(confidence), int(window))
            if column_key not in average_panel.columns:
                continue

            average_series = average_panel[column_key].dropna().sort_index()
            horizon_count = count_panel[column_key].reindex(average_series.index).astype(int)
            customdata = np.column_stack(
                [
                    np.full(len(average_series.index), expected_rate, dtype=float),
                    average_series.to_numpy(dtype=float) - expected_rate,
                    horizon_count.to_numpy(dtype=int),
                ]
            )

            fig.add_trace(
                go.Scatter(
                    x=average_series.index,
                    y=average_series,
                    mode='lines',
                    name=f'{int(window)}-session lookback',
                    legendgroup=f'window-{int(window)}',
                    showlegend=row_index == 1,
                    line=dict(color=trade_range_breach_average_colors.get(int(window), '#e2e8f0'), width=2.25),
                    customdata=customdata,
                    hovertemplate=(
                        'Date: %{x|%Y-%m-%d}'
                        f'<br>Session Lookback: {int(window)}-session'
                        f'<br>Confidence: {float(confidence):.0%}'
                        '<br>Average Either-Side Breach Rate: %{y:.2%}'
                        '<br>Expected Either-Side Breach Rate: %{customdata[0]:.2%}'
                        '<br>Average Excess Breach Rate: %{customdata[1]:.2%}'
                        '<br>Horizons Averaged: %{customdata[2]}'
                        '<extra></extra>'
                    ),
                ),
                row=row_index,
                col=1,
            )

        fig.add_hline(
            y=expected_rate,
            row=row_index,
            col=1,
            line_dash='dot',
            line_color='#94a3b8',
            line_width=1.5,
            annotation_text=f'Expected {expected_rate:.2%}',
            annotation_position='top right',
            annotation_font=dict(size=10, color='#94a3b8'),
        )
        fig.update_yaxes(title_text='Average Rolling Breach Rate', tickformat='.2%', row=row_index, col=1)

    fig.update_xaxes(title_text='Date', row=len(trade_range_breach_average_confidences), col=1)
    fig.update_layout(
        title=(
            f'{ticker_str} Average Either-Side Rolling Breach Rate Across Holding Horizons '
            f'(Horizons {min(trade_range_breach_average_horizons)}-{max(trade_range_breach_average_horizons)}; '
            f'Lookbacks {" / ".join(str(window) for window in trade_range_breach_average_windows)})'
        ),
        template='plotly_dark',
        height=360 * len(trade_range_breach_average_confidences) + 120,
        margin={'t': 120, 'r': 40, 'b': 70, 'l': 95},
        hovermode='x unified',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0.0),
    )
    return fig


trade_range_breach_average_panel, trade_range_breach_average_count_panel, trade_range_breach_average_support_text = (
    _build_trade_range_breach_average_panel()
)
trade_range_breach_average_fig = _build_trade_range_breach_average_figure(
    trade_range_breach_average_panel,
    trade_range_breach_average_count_panel,
)
globals()['trade_range_breach_average_panel'] = trade_range_breach_average_panel.copy()
globals()['trade_range_breach_average_count_panel'] = trade_range_breach_average_count_panel.copy()
globals()['trade_range_breach_average_support_text'] = trade_range_breach_average_support_text
globals()['trade_range_breach_average_fig'] = trade_range_breach_average_fig

print(trade_range_breach_average_support_text)
display(trade_range_breach_average_fig)


In [ ]:
# Block 8: backtest fixed payout structures against the 95% long confidence-interval floor

from plotly.subplots import make_subplots

strategy_payout_options = [(float(risk_dollars), float(100 - risk_dollars)) for risk_dollars in range(10, 100, 10)]
strategy_default_payout = (70.0, 30.0)
strategy_confidence = 0.95
strategy_rolling_pnl_window = 21


def format_strategy_payout_label(risk_dollars, reward_dollars):
    return f'Risk ${risk_dollars:,.0f} / Reward ${reward_dollars:,.0f}'


def format_strategy_payout_short_label(risk_dollars, reward_dollars):
    return f'{risk_dollars:,.0f}/{reward_dollars:,.0f}'


def build_fixed_payout_trade_profile(strategy_label, metrics_by_confidence, *, confidence, risk_dollars, reward_dollars, window_label, rolling_window):
    metric_set = metrics_by_confidence.get(confidence)
    if metric_set is None:
        raise KeyError(f'{strategy_label} does not contain a {confidence:.0%} long interval-floor series.')

    long_interval_floor = pd.Series(metric_set['lower_interval_price']).dropna()
    session_open = pd.Series(metric_set['session_open']).reindex(long_interval_floor.index)
    session_close = pd.Series(metric_set['session_close']).reindex(long_interval_floor.index)
    session_returns = pd.Series(metric_set['session_returns']).reindex(long_interval_floor.index)

    trade_frame = pd.DataFrame({
        'Open': session_open,
        'Close': session_close,
        'Close-to-Close Return': session_returns,
        'Long 95% Interval Floor': long_interval_floor,
    }).dropna()
    if trade_frame.empty:
        raise ValueError(f'{strategy_label} does not contain any fully aligned historical trades to evaluate.')

    breakeven_win_rate = risk_dollars / (risk_dollars + reward_dollars)
    payout_label = format_strategy_payout_label(risk_dollars, reward_dollars)

    trade_frame['Win'] = trade_frame['Close'].ge(trade_frame['Long 95% Interval Floor'])
    trade_frame['Outcome'] = np.where(trade_frame['Win'], 'Win', 'Loss')
    trade_frame['Trade PnL'] = np.where(trade_frame['Win'], reward_dollars, -risk_dollars)
    trade_frame['Rolling PnL'] = trade_frame['Trade PnL'].rolling(rolling_window, min_periods=1).sum()
    trade_frame['Rolling Win Rate'] = trade_frame['Win'].rolling(rolling_window, min_periods=1).mean()
    trade_frame['Cumulative PnL'] = trade_frame['Trade PnL'].cumsum()
    trade_frame['Running Peak PnL'] = trade_frame['Cumulative PnL'].cummax()
    trade_frame['Drawdown'] = trade_frame['Cumulative PnL'] - trade_frame['Running Peak PnL']
    trade_frame['Payout Structure'] = payout_label

    gross_profit = float(trade_frame.loc[trade_frame['Trade PnL'] > 0, 'Trade PnL'].sum())
    gross_loss = float(-trade_frame.loc[trade_frame['Trade PnL'] < 0, 'Trade PnL'].sum())
    profit_factor = np.nan if gross_loss == 0 else gross_profit / gross_loss

    summary = {
        'Payout Structure': payout_label,
        'Strategy': strategy_label,
        'Session Lookback Window': window_label,
        'Confidence': f'{confidence:.0%}',
        'Trades': int(len(trade_frame)),
        'Wins': int(trade_frame['Win'].sum()),
        'Losses': int((~trade_frame['Win']).sum()),
        'Win Rate': float(trade_frame['Win'].mean()),
        'Breakeven Win Rate': float(breakeven_win_rate),
        'Average Trade PnL': float(trade_frame['Trade PnL'].mean()),
        'Latest Rolling PnL': float(trade_frame['Rolling PnL'].iloc[-1]),
        'Total PnL': float(trade_frame['Trade PnL'].sum()),
        'Profit Factor': profit_factor,
        'Worst Drawdown': float(trade_frame['Drawdown'].min()),
    }

    return {
        'label': strategy_label,
        'summary': summary,
        'trade_frame': trade_frame,
    }


strategy_include_garch = 'garch_trade_history_context' in globals()
if not strategy_include_garch:
    print('Run Block 7 first if you want the GARCH strategy included in this fixed-payout evaluation.')


def build_strategy_profiles_for_payout(risk_dollars, reward_dollars):
    strategy_profiles = []
    for window in trade_range_history_context['windows']:
        strategy_profiles.append(
            build_fixed_payout_trade_profile(
                strategy_label=f'Current Method ({window}-Session)',
                metrics_by_confidence=trade_range_history_context['metrics_by_window'][window],
                confidence=strategy_confidence,
                risk_dollars=risk_dollars,
                reward_dollars=reward_dollars,
                window_label=window,
                rolling_window=strategy_rolling_pnl_window,
            )
        )

    if strategy_include_garch:
        garch_window_label = garch_trade_history_context.get('window', trade_range_default_window)
        strategy_profiles.append(
            build_fixed_payout_trade_profile(
                strategy_label=f'GARCH(1,1) Normal ({garch_window_label}-Session)',
                metrics_by_confidence=garch_trade_history_context['metrics_by_confidence'],
                confidence=strategy_confidence,
                risk_dollars=risk_dollars,
                reward_dollars=reward_dollars,
                window_label=garch_window_label,
                rolling_window=strategy_rolling_pnl_window,
            )
        )

    return strategy_profiles


strategy_default_payout_label = format_strategy_payout_label(*strategy_default_payout)
strategy_profiles_by_payout = {}
strategy_summary_by_payout = {}
strategy_backtest_trade_logs_by_payout = {}
strategy_backtest_equity_curve_by_payout = {}
strategy_backtest_rolling_pnl_by_payout = {}

for risk_dollars, reward_dollars in strategy_payout_options:
    payout_label = format_strategy_payout_label(risk_dollars, reward_dollars)
    strategy_profiles = build_strategy_profiles_for_payout(risk_dollars, reward_dollars)
    strategy_profiles_by_payout[payout_label] = strategy_profiles
    strategy_summary_by_payout[payout_label] = pd.DataFrame([profile['summary'] for profile in strategy_profiles])
    strategy_backtest_trade_logs_by_payout[payout_label] = {
        profile['label']: profile['trade_frame'].copy()
        for profile in strategy_profiles
    }
    strategy_backtest_equity_curve_by_payout[payout_label] = pd.concat(
        {profile['label']: profile['trade_frame']['Cumulative PnL'] for profile in strategy_profiles},
        axis=1,
    ).sort_index()
    strategy_backtest_rolling_pnl_by_payout[payout_label] = pd.concat(
        {profile['label']: profile['trade_frame']['Rolling PnL'] for profile in strategy_profiles},
        axis=1,
    ).sort_index()

strategy_summary_lookup = pd.concat(strategy_summary_by_payout.values(), ignore_index=True)
formatted_strategy_summary_lookup = strategy_summary_lookup.copy()
for column in ('Win Rate', 'Breakeven Win Rate'):
    if column in formatted_strategy_summary_lookup.columns:
        formatted_strategy_summary_lookup[column] = formatted_strategy_summary_lookup[column].map(
            lambda value: f'{value:.2%}' if pd.notna(value) else value
        )
for column in ('Average Trade PnL', 'Latest Rolling PnL', 'Total PnL', 'Worst Drawdown'):
    if column in formatted_strategy_summary_lookup.columns:
        formatted_strategy_summary_lookup[column] = formatted_strategy_summary_lookup[column].map(
            lambda value: f'${value:,.2f}' if pd.notna(value) else value
        )
if 'Profit Factor' in formatted_strategy_summary_lookup.columns:
    formatted_strategy_summary_lookup['Profit Factor'] = formatted_strategy_summary_lookup['Profit Factor'].map(
        lambda value: 'N/A' if pd.isna(value) else f'{value:.2f}'
    )

display(formatted_strategy_summary_lookup)

strategy_backtest_trade_logs = strategy_backtest_trade_logs_by_payout[strategy_default_payout_label]
strategy_backtest_equity_curve = strategy_backtest_equity_curve_by_payout[strategy_default_payout_label]
strategy_backtest_rolling_pnl = strategy_backtest_rolling_pnl_by_payout[strategy_default_payout_label]
strategy_trade_log_preview = pd.concat(
    [
        profile['trade_frame'].assign(Strategy=profile['label'])
        for profile in strategy_profiles_by_payout[strategy_default_payout_label]
    ],
    axis=0,
).reset_index(names='Date')
strategy_trade_log_preview = strategy_trade_log_preview[['Date', 'Payout Structure', 'Strategy', 'Close', 'Long 95% Interval Floor', 'Outcome', 'Trade PnL', 'Rolling PnL', 'Rolling Win Rate', 'Cumulative PnL', 'Drawdown']]
display(strategy_trade_log_preview.tail(15))

strategy_backtest_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.48, 0.52],
    subplot_titles=(
        f'{strategy_rolling_pnl_window}-Trade Rolling PnL',
        'Cumulative PnL',
    ),
)

strategy_trace_indices_by_payout = {}
for risk_dollars, reward_dollars in strategy_payout_options:
    payout_label = format_strategy_payout_label(risk_dollars, reward_dollars)
    payout_short_label = format_strategy_payout_short_label(risk_dollars, reward_dollars)
    strategy_profiles = strategy_profiles_by_payout[payout_label]
    strategy_trace_indices_by_payout[payout_label] = []
    is_default_payout = payout_label == strategy_default_payout_label

    for profile in strategy_profiles:
        trade_frame = profile['trade_frame']

        rolling_trace_index = len(strategy_backtest_fig.data)
        strategy_backtest_fig.add_trace(
            go.Scatter(
                x=trade_frame.index,
                y=trade_frame['Rolling PnL'],
                mode='lines',
                name=profile['label'],
                legendgroup=profile['label'],
                showlegend=True,
                visible=is_default_payout,
                customdata=np.column_stack([
                    trade_frame['Trade PnL'],
                    trade_frame['Rolling Win Rate'],
                    trade_frame['Cumulative PnL'],
                ]),
                hovertemplate=(
                    f'Payout: {payout_short_label}'
                    '<br>Date: %{x|%Y-%m-%d}'
                    '<br>Rolling PnL: $%{y:,.2f}'
                    '<br>Trade PnL: $%{customdata[0]:,.2f}'
                    '<br>Rolling Win Rate: %{customdata[1]:.2%}'
                    '<br>Cumulative PnL: $%{customdata[2]:,.2f}'
                    '<extra></extra>'
                ),
            ),
            row=1,
            col=1,
        )
        strategy_trace_indices_by_payout[payout_label].append(rolling_trace_index)

        cumulative_trace_index = len(strategy_backtest_fig.data)
        strategy_backtest_fig.add_trace(
            go.Scatter(
                x=trade_frame.index,
                y=trade_frame['Cumulative PnL'],
                mode='lines',
                name=profile['label'],
                legendgroup=profile['label'],
                showlegend=False,
                visible=is_default_payout,
                customdata=np.column_stack([
                    trade_frame['Trade PnL'],
                    trade_frame['Rolling PnL'],
                    trade_frame['Drawdown'],
                ]),
                hovertemplate=(
                    f'Payout: {payout_short_label}'
                    '<br>Date: %{x|%Y-%m-%d}'
                    '<br>Cumulative PnL: $%{y:,.2f}'
                    '<br>Trade PnL: $%{customdata[0]:,.2f}'
                    '<br>Rolling PnL: $%{customdata[1]:,.2f}'
                    '<br>Drawdown: $%{customdata[2]:,.2f}'
                    '<extra></extra>'
                ),
            ),
            row=2,
            col=1,
        )
        strategy_trace_indices_by_payout[payout_label].append(cumulative_trace_index)

strategy_backtest_fig.add_hline(
    y=0,
    line_dash='dash',
    line_color='rgba(248, 250, 252, 0.45)',
    line_width=1,
    row=1,
    col=1,
)
strategy_backtest_fig.add_hline(
    y=0,
    line_dash='dash',
    line_color='rgba(248, 250, 252, 0.45)',
    line_width=1,
    row=2,
    col=1,
)

strategy_dropdown_buttons = []
strategy_total_trace_count = len(strategy_backtest_fig.data)
for risk_dollars, reward_dollars in strategy_payout_options:
    payout_label = format_strategy_payout_label(risk_dollars, reward_dollars)
    visible_mask = [False] * strategy_total_trace_count
    for trace_index in strategy_trace_indices_by_payout[payout_label]:
        visible_mask[trace_index] = True

    strategy_dropdown_buttons.append(
        dict(
            label=format_strategy_payout_short_label(risk_dollars, reward_dollars),
            method='update',
            args=[
                {'visible': visible_mask},
                {
                    'title': lineChartPlotter._header_title(
                        f'{ticker_str} Fixed Payout Strategy vs 95% Long Interval Floor ({payout_label})'
                    )
                },
            ],
        )
    )

strategy_backtest_fig.update_layout(
    title=lineChartPlotter._header_title(
        f'{ticker_str} Fixed Payout Strategy vs 95% Long Interval Floor ({strategy_default_payout_label})'
    ),
    height=950,
    margin=lineChartPlotter._header_margin(),
    template='plotly_dark',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    updatemenus=[
        dict(
            type='dropdown',
            direction='down',
            buttons=strategy_dropdown_buttons,
            showactive=True,
            active=strategy_payout_options.index(strategy_default_payout),
            x=1.0,
            xanchor='right',
            y=1.18,
            yanchor='top',
            bgcolor='rgba(15, 23, 42, 0.95)',
            bordercolor='rgba(148, 163, 184, 0.45)',
            font=dict(size=11),
        )
    ],
    annotations=list(strategy_backtest_fig.layout.annotations) + [
        dict(
            x=1.0,
            y=1.205,
            xref='paper',
            yref='paper',
            xanchor='right',
            yanchor='bottom',
            showarrow=False,
            text='Risk / Reward',
            font=dict(size=11, color='rgba(226, 232, 240, 0.92)'),
        )
    ],
)
strategy_backtest_fig.update_xaxes(title_text='Date', row=2, col=1)
strategy_backtest_fig.update_yaxes(title_text=f'{strategy_rolling_pnl_window}-Trade Rolling PnL', tickprefix='$', row=1, col=1)
strategy_backtest_fig.update_yaxes(title_text='Cumulative PnL', tickprefix='$', row=2, col=1)
show_plotly_figure(strategy_backtest_fig)


In [ ]:
# Block 9: compare annualized GARCH-family volatility models to annualized rolling volatility estimators

import sys

def _purge_stale_modules(prefixes):
    for prefix in prefixes:
        matching_modules = [
            name
            for name in list(sys.modules)
            if name == prefix or name.startswith(f"{prefix}.")
        ]
        for module_name in matching_modules:
            sys.modules.pop(module_name, None)

try:
    from arch import arch_model
except ModuleNotFoundError as exc:
    if exc.name != "arch":
        raise
    raise ImportError(
        "Block 10 requires the 'arch' package. Install it in the notebook kernel environment with `pip install arch`."
    ) from exc
except Exception:
    _purge_stale_modules(("arch", "matplotlib"))
    try:
        from arch import arch_model
    except ModuleNotFoundError as exc:
        if exc.name != "arch":
            raise
        raise ImportError(
            "Block 10 requires the 'arch' package. Install it in the notebook kernel environment with `pip install arch`."
        ) from exc
    except Exception as exc:
        raise RuntimeError(
            "Block 10 could not import `arch` because the notebook kernel is holding a stale matplotlib state. Restart the kernel and rerun Block 2 if this persists."
        ) from exc

close_series = ticker["Close"]
ohlc_frame = ticker[["Open", "High", "Low", "Close"]].copy()
returns = close_series.pct_change()
garch_input = returns.dropna() * 100

log_hl = np.log(ohlc_frame["High"] / ohlc_frame["Low"])
log_ho = np.log(ohlc_frame["High"] / ohlc_frame["Open"])
log_lo = np.log(ohlc_frame["Low"] / ohlc_frame["Open"])
log_co = np.log(ohlc_frame["Close"] / ohlc_frame["Open"])
log_oc = np.log(ohlc_frame["Open"] / ohlc_frame["Close"].shift(1))
log_hc = np.log(ohlc_frame["High"] / ohlc_frame["Close"])
log_lc = np.log(ohlc_frame["Low"] / ohlc_frame["Close"])

garman_klass_variance = 0.5 * (log_hl ** 2) - ((2 * np.log(2)) - 1) * (log_co ** 2)
parkinson_variance = (log_hl ** 2) / (4 * np.log(2))
rs_variance = (log_hc * log_ho) + (log_lc * log_lo)

volatility_model_specs = [
    ("GARCH(1,1)", dict(vol="GARCH", p=1, q=1, o=0), "#111111", "solid"),
    ("EGARCH(1,1)", dict(vol="EGARCH", p=1, o=1, q=1), "#d62728", "dash"),
    ("GJR-GARCH(1,1)", dict(vol="GARCH", p=1, o=1, q=1), "#2ca02c", "dot"),
]

rolling_realized_vol_specs = [
    ("Close-to-Close", "close-to-close", "#1f77b4", "solid"),
    ("Parkinson", "parkinson", "#9467bd", "dash"),
    ("Yang-Zhang", "yang-zhang", "#ff7f0e", "dot"),
    ("Garman-Klass", "garman-klass", "#8c564b", "dashdot"),
    ("Rogers-Satchell", "rogers-satchell", "#17becf", "longdash"),
]

ewma_realized_vol_specs = [
    ("EWMA Close-to-Close", "close-to-close", "#1f77b4", "longdashdot"),
    ("EWMA Parkinson", "parkinson", "#9467bd", "longdashdot"),
    ("EWMA Yang-Zhang", "yang-zhang", "#ff7f0e", "longdashdot"),
    ("EWMA Garman-Klass", "garman-klass", "#8c564b", "longdashdot"),
    ("EWMA Rogers-Satchell", "rogers-satchell", "#17becf", "longdashdot"),
]

annualized_model_vols = {}
for model_name, model_kwargs, _, _ in volatility_model_specs:
    model_fit = arch_model(
        garch_input,
        mean="Zero",
        dist="normal",
        rescale=False,
        **model_kwargs,
    ).fit(disp="off")
    annualized_model_vol = (model_fit.conditional_volatility / 100.0) * np.sqrt(252)
    annualized_model_vol.name = f"Annualized {model_name}"
    annualized_model_vols[model_name] = annualized_model_vol

def compute_rolling_realized_vol_series(window, method):
    if method == "close-to-close":
        return returns.rolling(window).std() * np.sqrt(252)

    volatility_frame = rolling.volatility(ohlc_frame, windows=(window,), method=method)
    return volatility_frame.iloc[:, 0]

def compute_ewma_realized_vol_series(window, method):
    alpha = 2.0 / (window + 1.0)

    if method == "close-to-close":
        ewma_variance = returns.pow(2).ewm(alpha=alpha, adjust=False, min_periods=window).mean()
        return np.sqrt(ewma_variance * 252)

    if method == "parkinson":
        ewma_variance = parkinson_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        return np.sqrt(ewma_variance * 252)

    if method == "garman-klass":
        ewma_variance = garman_klass_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        return np.sqrt(ewma_variance * 252)

    if method == "rogers-satchell":
        ewma_variance = rs_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        return np.sqrt(ewma_variance * 252)

    if method == "yang-zhang":
        if window < 2:
            return pd.Series(np.nan, index=ohlc_frame.index)

        k = 0.34 / (1.34 + ((window + 1) / (window - 1)))
        overnight_variance = log_oc.ewm(alpha=alpha, adjust=False, min_periods=window).var(bias=False)
        open_to_close_variance = log_co.ewm(alpha=alpha, adjust=False, min_periods=window).var(bias=False)
        rs_component = rs_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean()
        yz_variance = (
            overnight_variance
            + (k * open_to_close_variance)
            + ((1 - k) * rs_component)
        ).clip(lower=0)
        return np.sqrt(yz_variance * 252)

    raise ValueError(f"Unsupported EWMA volatility method: {method}")

def smooth_annualized_model_vol(model_vol, window):
    # Smooth model variance over the same horizon used by the trailing realized-vol estimator.
    return np.sqrt(model_vol.pow(2).rolling(window).mean())

volatility_term_order = [term for term in time_frame_map if time_frame_map.get(term) is not None]
default_vol_term = 'long' if 'long' in volatility_term_order else max(
    volatility_term_order,
    key=lambda term: int(time_frame_map[term]),
)

vol_model_fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=(
        "Annualized GARCH Family vs Close-to-Close Volatility (Raw + Smoothed)",
        "Annualized Rolling and EWMA Volatility Estimators",
        "Spread vs Close-to-Close Annualized Volatility",
        "Annualized Rolling Minus EWMA Volatility Estimators",
    ),
)
term_trace_bounds = {}
term_ranges = {}

for term in volatility_term_order:
    window = int(time_frame_map[term])
    rolling_realized_vol_map = {
        label: compute_rolling_realized_vol_series(window, method)
        for label, method, _, _ in rolling_realized_vol_specs
    }
    ewma_realized_vol_map = {
        label: compute_ewma_realized_vol_series(window, method)
        for label, method, _, _ in ewma_realized_vol_specs
    }
    baseline_vol = rolling_realized_vol_map["Close-to-Close"]

    visible = term == default_vol_term
    term_trace_start = len(vol_model_fig.data)

    vol_model_fig.add_trace(
        go.Scatter(
            x=baseline_vol.index,
            y=baseline_vol,
            mode="lines",
            name=f"Close-to-Close ({window}-day)",
            line=dict(color="#1f77b4", width=2),
            visible=visible,
            showlegend=visible,
            legendgroup="close-to-close-baseline",
        ),
        row=1,
        col=1,
    )

    for model_name, _, color, dash in volatility_model_specs:
        model_vol = annualized_model_vols[model_name]
        smoothed_model_vol = smooth_annualized_model_vol(model_vol, window)
        model_spread = (model_vol - baseline_vol).dropna()
        smoothed_model_spread = (smoothed_model_vol - baseline_vol).dropna()

        vol_model_fig.add_trace(
            go.Scatter(
                x=model_vol.index,
                y=model_vol,
                mode="lines",
                name=f"Annualized {model_name}",
                line=dict(color=color, width=2, dash=dash),
                visible=visible,
                showlegend=visible,
                legendgroup=model_name,
            ),
            row=1,
            col=1,
        )

        vol_model_fig.add_trace(
            go.Scatter(
                x=smoothed_model_vol.index,
                y=smoothed_model_vol,
                mode="lines",
                name=f"{model_name} Smoothed ({window}-day)",
                line=dict(color=color, width=3, dash="longdash"),
                opacity=0.9,
                visible=visible,
                showlegend=visible,
                legendgroup=f"{model_name}-smoothed",
            ),
            row=1,
            col=1,
        )

        vol_model_fig.add_trace(
            go.Scatter(
                x=model_spread.index,
                y=model_spread,
                mode="lines",
                name=f"{model_name} Spread",
                line=dict(color=color, width=2, dash=dash),
                visible=visible,
                showlegend=False,
                legendgroup=f"{model_name}-spread",
            ),
            row=3,
            col=1,
        )

        vol_model_fig.add_trace(
            go.Scatter(
                x=smoothed_model_spread.index,
                y=smoothed_model_spread,
                mode="lines",
                name=f"{model_name} Smoothed Spread",
                line=dict(color=color, width=3, dash="longdash"),
                opacity=0.9,
                visible=visible,
                showlegend=False,
                legendgroup=f"{model_name}-smoothed-spread",
            ),
            row=3,
            col=1,
        )

    for label, _, color, dash in rolling_realized_vol_specs:
        realized_vol = rolling_realized_vol_map[label]

        vol_model_fig.add_trace(
            go.Scatter(
                x=realized_vol.index,
                y=realized_vol,
                mode="lines",
                name=f"{label} ({window}-day)",
                line=dict(color=color, width=2, dash=dash),
                visible=visible,
                showlegend=False,
                legendgroup=f"rolling-realized-{label}",
            ),
            row=2,
            col=1,
        )

        if label != "Close-to-Close":
            realized_spread = (realized_vol - baseline_vol).dropna()
            vol_model_fig.add_trace(
                go.Scatter(
                    x=realized_spread.index,
                    y=realized_spread,
                    mode="lines",
                    name=f"{label} Spread",
                    line=dict(color=color, width=2, dash=dash),
                    visible=visible,
                    showlegend=False,
                    legendgroup=f"rolling-realized-spread-{label}",
                ),
                row=3,
                col=1,
            )

    for label, method, color, dash in ewma_realized_vol_specs:
        ewma_vol = ewma_realized_vol_map[label]

        vol_model_fig.add_trace(
            go.Scatter(
                x=ewma_vol.index,
                y=ewma_vol,
                mode="lines",
                name=f"{label} ({window}-day)",
                line=dict(color=color, width=2, dash=dash),
                opacity=0.9,
                visible=visible,
                showlegend=False,
                legendgroup=f"ewma-realized-{label}",
            ),
            row=2,
            col=1,
        )

        ewma_spread = (ewma_vol - baseline_vol).dropna()
        vol_model_fig.add_trace(
            go.Scatter(
                x=ewma_spread.index,
                y=ewma_spread,
                mode="lines",
                name=f"{label} Spread",
                line=dict(color=color, width=2, dash=dash),
                opacity=0.9,
                visible=visible,
                showlegend=False,
                legendgroup=f"ewma-realized-spread-{label}",
            ),
            row=3,
            col=1,
        )

    for (rolling_label, _, color, _), (ewma_label, _, _, _) in zip(rolling_realized_vol_specs, ewma_realized_vol_specs):
        realized_minus_ewma = (rolling_realized_vol_map[rolling_label] - ewma_realized_vol_map[ewma_label]).dropna()

        vol_model_fig.add_trace(
            go.Scatter(
                x=realized_minus_ewma.index,
                y=realized_minus_ewma,
                mode="lines",
                name=f"{rolling_label} Minus {ewma_label} ({window}-day)",
                line=dict(color=color, width=3),
                opacity=0.9,
                visible=visible,
                showlegend=False,
                legendgroup=f"realized-minus-ewma-{rolling_label}",
            ),
            row=4,
            col=1,
        )

    term_trace_bounds[term] = (term_trace_start, len(vol_model_fig.data))

    non_empty_series = [
        series
        for series in (
            [series.dropna() for series in rolling_realized_vol_map.values()]
            + [series.dropna() for series in ewma_realized_vol_map.values()]
            + [model_series.dropna() for model_series in annualized_model_vols.values()]
        )
        if not series.empty
    ]
    if non_empty_series:
        max_index = max(series.index.max() for series in non_empty_series)
        min_index = min(series.index.min() for series in non_empty_series)
        term_ranges[term] = [max(min_index, max_index - pd.DateOffset(years=3)), max_index]
    else:
        term_ranges[term] = None

vol_model_fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="rgba(120, 120, 120, 0.8)",
    row=3,
    col=1,
)
vol_model_fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="rgba(120, 120, 120, 0.8)",
    row=4,
    col=1,
)

vol_buttons = []
total_traces = len(vol_model_fig.data)
for term in volatility_term_order:
    visibility = [False] * total_traces
    start, end = term_trace_bounds[term]
    for trace_idx in range(start, end):
        visibility[trace_idx] = True

    layout_updates = {
        "title": {
            "text": f"{ticker_str} Volatility Models and Estimators ({time_frame_map[term]}-Day)",
            "x": 0.5,
            "xanchor": "center",
            "y": 0.97,
            "yanchor": "top",
        },
        "yaxis": {"title": "Annualized Volatility"},
        "yaxis2": {"title": "Annualized Volatility"},
        "yaxis3": {"title": "Spread vs Close-to-Close"},
        "yaxis4": {"title": "Rolling Minus EWMA"},
    }
    if term_ranges.get(term) is not None:
        layout_updates["xaxis"] = {"range": term_ranges[term]}
        layout_updates["xaxis2"] = {"range": term_ranges[term]}
        layout_updates["xaxis3"] = {"range": term_ranges[term]}
        layout_updates["xaxis4"] = {"range": term_ranges[term]}

    vol_buttons.append(
        dict(
            label=f"{term.title()} ({time_frame_map[term]})",
            method="update",
            args=[{"visible": visibility}, layout_updates],
        )
    )

available_ranges = [date_range for date_range in term_ranges.values() if date_range is not None]
if available_ranges:
    global_start = min(date_range[0] for date_range in available_ranges)
    global_end = max(date_range[1] for date_range in available_ranges)
    default_range = term_ranges[default_vol_term] or [global_start, global_end]
    vol_model_fig.update_xaxes(range=default_range, row=1, col=1)
    vol_model_fig.update_xaxes(range=default_range, row=2, col=1)
    vol_model_fig.update_xaxes(range=default_range, row=3, col=1)
    vol_model_fig.update_xaxes(range=default_range, row=4, col=1)
    time_range_menu = dict(
        type="dropdown",
        buttons=build_time_range_buttons(global_start, global_end, axis_count=4),
        x=0.28,
        xanchor="left",
        y=1.08,
        yanchor="top",
        showactive=True,
    )
else:
    time_range_menu = None

vol_model_fig.update_layout(
    template="plotly_white",
    height=1450,
    margin=dict(t=150),
    legend=dict(x=0.01, y=0.99),
    yaxis_title="Annualized Volatility",
    yaxis2_title="Annualized Volatility",
    yaxis3_title="Spread vs Close-to-Close",
    yaxis4_title="Rolling Minus EWMA",
    title=dict(
        text=f"{ticker_str} Volatility Models and Estimators ({time_frame_map[default_vol_term]}-Day)",
        x=0.5,
        xanchor="center",
        y=0.97,
        yanchor="top",
    ),
    updatemenus=[
        dict(
            type="dropdown",
            buttons=vol_buttons,
            x=0.0,
            xanchor="left",
            y=1.08,
            yanchor="top",
            showactive=True,
            active=volatility_term_order.index(default_vol_term),
        )
    ] + ([time_range_menu] if time_range_menu is not None else []),
)
show_plotly_figure(vol_model_fig)